<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 MiniMax H3 Turbo (INT8) - Video + Audio</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Text · Image · First/Last Frame · Reference → Video with synced stereo audio | 4-step PDMD / DMAD | Wan2GP + INT8 Tensor Core Kernels | LoRA support</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is MiniMax H3?

**MiniMax H3** generates video **and** native 32 kHz stereo audio together: speech with lip sync, sound effects and music, from text, a start image, first + last frames, or reference images of characters and objects. This notebook runs the pruned 20B model on free Kaggle **GPU T4 x2** through the **Wan2GP** engine.

| Feature | Details |
|----|----|
| Model | MiniMax H3 pruned 20B, INT8 ConvRot. Pick one in Step 1 (`H3_VARIANT`):<br>• **`pdmd`** (default): base H3 + the [PDMD](https://github.com/ZeamoxWang/pdmd) 4-step LoRA<br>• **`dmad`**: base H3 + the [DMAD](https://github.com/Yzmblog/DMAD) 4-step LoRA<br>• **`fused`**: [MATLOWAI fused turbo](https://huggingface.co/MATLOWAI/minimax-h3-fused-turbo-int8-convrot), Ref2VA (needed for Reference → Video)<br>• **`official`**: Ref2VA + the lightx2v 8-step LoRA<br>The distillation LoRAs run live and exact on top of the INT8 model (~0.8 GB VRAM per GPU, a few % slower) |
| Speed tricks | 4-step distillation · Comfy Kitchen INT8 tensor-core GEMMs · **tensor-parallel DiT**: every block split across both T4s so they compute at the same time · whole DiT resident in VRAM, no weight streaming |
| Text encoder | Qwen3-VL-32B GGUF Q4_K_M, pinned in RAM, fastest GEMM dtype picked at startup, cached per prompt |
| VAE | INT8 ConvRot video decoder + FP32 audio VAE |
| Modes | Text → Video, Image → Video, First + Last Frame (all variants); Reference → Video with up to 3 images + voice clone (`fused` / `official`) |
| LoRAs | 9 popular H3 LoRAs in a dropdown (realism, camera motion, 360° orbit, combat, physics, talking faces, animation styles) + any custom `.safetensors` URL / upload |
| Output | MP4 with audio, saved to `/kaggle/working/outputs` (Kaggle Output panel) |

### Quick Start
1. **Settings → Accelerator → GPU T4 x2**, and **turn on Internet** in the Settings sidebar
2. **Pick your model** with `H3_VARIANT` on the first line of Step 1. The default is `"pdmd"`; for Reference → Video use `"fused"` (see the table below)
3. Run all cells. Step 1 installs and downloads ~40 GB of weights (about 5-10 minutes). Step 2 loads the model (about 3-4 minutes) and starts the UI
4. Open the **Cloudflare** link (live progress bar) or the **Gradio** share link printed at the bottom of Step 2
5. Start with **480p · 5 s · 4 steps**. Put exact speech in **Spoken line** for lip-synced dialogue, and describe sounds in **Sound effects**
6. For **Reference → Video**, set `H3_VARIANT = "fused"` in Step 1, then add 1-3 reference images (and optionally a 2-15 s voice sample to clone)

> 💡 **Save the downloads:** after the first run, create a Kaggle Dataset from `/kaggle/tmp/h3_models` and attach it with **Add Input**. Step 1 finds the files there and skips the 40 GB download.

## ⚙️ Choose your model (first line of Step 1)

Set `H3_VARIANT` at the top of **Step 1** to one of these four, then run the cells. Step 1 downloads only the files that model needs.

| `H3_VARIANT` | Model | Text → Video | Image / First + Last Frame | Reference → Video + voice clone | Steps |
|---|---|:---:|:---:|:---:|:---:|
| **`"pdmd"`** ⭐ default | Base H3 + [PDMD](https://github.com/ZeamoxWang/pdmd) 4-step LoRA (ByteDance, 2026) | ✅ | ✅ | ❌ | 4 |
| **`"dmad"`** | Base H3 + [DMAD](https://github.com/Yzmblog/DMAD) 4-step LoRA (ByteDance, 2026) | ✅ | ✅ | ❌ | 4 |
| **`"fused"`** | [MATLOWAI fused turbo](https://huggingface.co/MATLOWAI/minimax-h3-fused-turbo-int8-convrot) (Ref2VA) | ✅ | ✅ | ✅ | 4 |
| **`"official"`** | Ref2VA + lightx2v 8-step turbo LoRA | ✅ | ✅ | ✅ | 8 |

* **PDMD / DMAD** are the newest 4-step distillations and aim for the best quality. They were trained on the base model, which has no Reference → Video.
* **Need reference images or a cloned voice?** Use `"fused"`.
* **Switching models:** change the line, then re-run Step 1 and Step 2. Files you already downloaded stay on disk.

In [ ]:
# Step 1: Environment, pinned Wan2GP, dependencies and model downloads

# ╔══════════════════════════════════════════════════════════════════════════════════════╗
# ║  👉 CHOOSE YOUR MODEL: "pdmd" (default) · "dmad" · "fused" · "official"              ║
# ║     pdmd / dmad: newest 4-step distillations; text, image, first + last frame        ║
# ║     fused / official: also Reference → Video + voice clone (see the table above)     ║
# ╚══════════════════════════════════════════════════════════════════════════════════════╝
H3_VARIANT = "pdmd"

import os
import sys
import time
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor

def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~23 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_h3.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()

# ---- Settings ----
WAN2GP_COMMIT = "b8b18f8114e432eea8f3d7e853a51dd91fa99571"  # Wan2GP, 1 Oct 2026 (the version this notebook was built on)
MODELS_DIR = "/kaggle/tmp/h3_models" if os.path.isdir("/kaggle") else os.path.abspath("h3_models")
LORA_DIR = "/kaggle/tmp/h3_loras" if os.path.isdir("/kaggle") else os.path.abspath("h3_loras")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no second copy of 40 GB in the xet cache
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("=" * 70)
print("🎬 MiniMax H3 Turbo INT8 - Video + Audio Generator")
print("📺 Created by: AIQUEST Academy | YouTube & X: @AIQuestAcademy")
print("=" * 70)
MODEL_MENU = {
    "pdmd": "Base H3 + PDMD 4-step LoRA       | text, image, first+last frame | 4 steps",
    "dmad": "Base H3 + DMAD 4-step LoRA       | text, image, first+last frame | 4 steps",
    "fused": "MATLOWAI fused turbo (Ref2VA)    | + Reference → Video, voice    | 4 steps",
    "official": "Ref2VA + lightx2v 8-step LoRA    | + Reference → Video, voice    | 8 steps",
}
if H3_VARIANT not in MODEL_MENU:
    raise SystemExit(f'❌ Unknown H3_VARIANT "{H3_VARIANT}". Set it to one of: ' + ", ".join(f'"{k}"' for k in MODEL_MENU))
print("🧠 Model (H3_VARIANT, first line of this cell):")
for key, text in MODEL_MENU.items():
    print(f"   {'👉' if key == H3_VARIANT else '  '} {key:<9} {text}")
print("   To switch: edit H3_VARIANT at the top of this cell and re-run Step 1 + Step 2.")
print("=" * 70)
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip().splitlines()
    for i, line in enumerate(smi):
        print(f"✅ GPU {i}: {line}")
    if len(smi) < 2:
        print("❌ This notebook needs 2 GPUs: Settings → Accelerator → GPU T4 x2")
except Exception:
    print("❌ No GPU found: Settings → Accelerator → GPU T4 x2")

# ---- 1. Wan2GP, pinned to a tested commit ----
t0 = time.time()
if not os.path.isdir("Wan2GP/.git"):
    print("\n📥 Fetching Wan2GP...")
    subprocess.run(["git", "init", "-q", "Wan2GP"], check=True)
    subprocess.run(["git", "-C", "Wan2GP", "remote", "add", "origin", "https://github.com/DeepBeepMeep/Wan2GP.git"], check=True)
subprocess.run(["git", "-C", "Wan2GP", "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT], check=True)
subprocess.run(["git", "-C", "Wan2GP", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)  # also undoes stale edits on re-runs
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]} ({time.time() - t0:.0f}s)")

# ---- 2. Dependencies: only what the H3 path imports ----
# (Wan2GP's full requirements.txt pins an onnxruntime nightly that is no longer on PyPI.)
print("📦 Installing dependencies...")
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5",
                "mmgp==3.8.2", "comfy-kitchen==0.2.35", "transformers==4.54.0", "diffusers==0.36.0", "gguf==0.17.1", "optimum-quanto", "accelerate>=1.1.1", "gradio==5.29.0", "huggingface_hub[hf_xet]", "hf_xet>=1.5.2", "sentencepiece", "soundfile"], check=True)
print(f"✅ Dependencies installed ({time.time() - t0:.0f}s)")

# ---- 3. Model weights (parallel; files from an attached Kaggle Dataset are reused) ----
H3 = "DeepBeepMeep/MiniMax-H3"
TE = "Qwen3-VL-32B-Instruct"
BASE_DIT = (H3, "MiniMax-H3-FL2VA-pruned_rank8_int8_convrot.safetensors", 21.06)  # the base T2VA / FL2VA transformer
if H3_VARIANT == "pdmd":  # PDMD (UCSD / ByteDance): https://github.com/ZeamoxWang/pdmd
    DIT, TURBO_LORA = BASE_DIT, ("pdmd2026/pdmd_4NFE_lora", "lora_model_0.safetensors", 1.38)
elif H3_VARIANT == "dmad":  # DMAD (TAMU / ByteDance): https://github.com/Yzmblog/DMAD
    DIT, TURBO_LORA = BASE_DIT, ("ZhengmingYu/DMAD", "minimax_h3/dmad_minimax_h3_4step_full_critic.safetensors", 1.38)
elif H3_VARIANT == "fused":
    DIT = ("MATLOWAI/minimax-h3-fused-turbo-int8-convrot",
           "diffusion_models/minimax_h3_fused_refdelta_r1024_turbo8_mystic07_int8_convrot.safetensors", 20.98)
    TURBO_LORA = None
elif H3_VARIANT == "official":
    DIT = (H3, "MiniMax-H3-Ref2VA-pruned_rank8_int8_convrot.safetensors", 21.06)
    TURBO_LORA = (H3, "loras/minimax_h3_lightx2v_ref2v_turbo_8step_alpha8_v1.0_768p_bf16.safetensors", 1.38)

BIG = [DIT,
       (H3, f"{TE}/qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf", 14.58),
       (H3, "minimax_h3/minimax_h3_video_vae_int8_convrot.safetensors", 2.81),
       (H3, "MiniMax-H3-audio_vae_fp32.safetensors", 0.61)]
SMALL = [(H3, f"{TE}/{name}", 0.0) for name in
         ("config.json", "tokenizer.json", "tokenizer_config.json", "vocab.json", "preprocessor_config.json", "chat_template.json")]
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)

attached = {}
if os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            attached.setdefault(f, os.path.join(root, f))


def fetch(item, local_dir=MODELS_DIR):
    from huggingface_hub import hf_hub_download
    repo, filename, size_gb = item
    dest = os.path.join(local_dir, filename)
    if os.path.exists(dest):
        return f"✓ {filename}"
    source = attached.get(os.path.basename(filename))
    if source and os.path.getsize(source) > 0:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        os.symlink(source, dest)
        return f"✓ {filename} (from attached dataset)"
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=repo, filename=filename, local_dir=local_dir)
            return f"✅ {filename}" + (f" ({size_gb:.1f} GB)" if size_gb else "")
        except Exception as e:
            if attempt == 3:
                raise
            print(f"  ⚠️ {filename}: {e} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)


def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total


todo = [item for item in BIG + ([TURBO_LORA] if TURBO_LORA else [])
        if not os.path.exists(os.path.join(MODELS_DIR if item is not TURBO_LORA else LORA_DIR, item[1]))
        and os.path.basename(item[1]) not in attached]
need_gb = sum(size for _, _, size in todo)
try:  # exact sizes for an exact progress bar (the sizes above are rounded)
    from huggingface_hub import HfApi
    exact = 0
    for repo in {item[0] for item in todo}:
        files = [item[1] for item in todo if item[0] == repo]
        exact += sum(info.size or 0 for info in HfApi().get_paths_info(repo, files))
    need_gb = exact / 1e9 if exact else need_gb
except Exception:
    pass
free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
print(f"\n🚀 Downloading {need_gb:.1f} GB of weights to {MODELS_DIR} ({free_gb:.0f} GB free)...")
if need_gb > free_gb - 2:
    raise SystemExit(f"❌ Not enough disk space: need {need_gb:.0f} GB, {free_gb:.0f} GB free.")
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]


def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def say(message):  # a normal line printed above the live bar
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)


def monitor():
    # One live line: GB done / total, %, smoothed speed, ETA. hf_xet writes into partial files, so bytes on disk
    # (minus what was already there) is the real progress.
    base = folder_bytes(MODELS_DIR) + folder_bytes(LORA_DIR)
    total = max(need_gb * 1e9, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) + folder_bytes(LORA_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"  ⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True


threading.Thread(target=monitor, daemon=True).start()
t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=6) as pool:
        jobs = [pool.submit(fetch, item) for item in BIG + SMALL]
        if TURBO_LORA:
            jobs.append(pool.submit(fetch, TURBO_LORA, LORA_DIR))
        for job in jobs:
            say("  " + job.result())
finally:
    done.set()
    time.sleep(1.1)
say(f"  ✅ {need_gb:.1f} GB downloaded in {fmt_time(time.time() - t0)}"
    + (f" (avg {need_gb * 1e3 / max(time.time() - t0, 1):.0f} MB/s)" if need_gb else ""))
for cache in (os.path.join(MODELS_DIR, ".cache"), os.path.join(LORA_DIR, ".cache")):
    shutil.rmtree(cache, ignore_errors=True)

# Settings for Step 2's engine (also saved to a file, so Step 2 can be re-run alone after a kernel restart)
import json
H3_SETTINGS = {"H3_VARIANT": H3_VARIANT, "H3_MODELS_DIR": MODELS_DIR, "H3_LORA_DIR": LORA_DIR, "H3_DIT_FILE": DIT[1],
               "H3_TURBO_LORA": TURBO_LORA[1] if TURBO_LORA else ""}
os.environ.update(H3_SETTINGS)
with open("h3_settings.json", "w") as f:
    json.dump(H3_SETTINGS, f)
print("\n" + "=" * 70)
print(f"✅ All weights ready in {time.time() - t0:.0f}s | disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB")
print("=" * 70)

In [ ]:
# Step 2: High-performance engine + branded Gradio UI
# Writes run_h3.py, then runs it: model load (~3-4 min) → public links.
import codecs
import os
import re
import sys
import subprocess

RUN_SCRIPT_CODE = r'''"""MiniMax H3 Turbo (INT8 ConvRot) video + audio engine for Kaggle GPU T4 x2 - AIQUEST Academy.

Step 2 of the notebook writes this file as run_h3.py and runs it. It drives Wan2GP's MiniMax H3 code directly:

* DiT: a pruned 20B H3 checkpoint in INT8 ConvRot. The default ("pdmd") is the base H3 transformer (FL2VA / T2VA
  partition) with the PDMD 4-step distillation LoRA applied live and exact ("dmad": the DMAD 4-step LoRA
  instead): text / image / first+last frame to video in 4 steps. "fused" is MATLOWAI's Ref2VA turbo file, the
  variant that also does reference-to-video.
* Every INT8 linear runs on Comfy Kitchen's INT8 tensor-core GEMM (W8A8 with the ConvRot Hadamard rotation).
* Activations stay in BF16 like upstream H3. Only attention runs in FP16 (Turing has no BF16 attention kernel);
  q and k are RMS-normalized there, so FP16 is safe. The residual stream never sees FP16, so it cannot overflow.
* Tensor parallel DiT: every block is split in half across the two T4s (attention heads and FFN columns), so both GPUs
  compute every block at the same time and only exchange the block's partial outputs. Nothing is streamed per step.
* Qwen3-VL-32B (GGUF Q4_K_M) and the VAEs are pinned in RAM and run by MMGP on GPU 1, unloaded before denoising.
"""
import copy
import faulthandler
import functools
import gc
import glob
import math
import os
import random
import re
import subprocess
import sys
import threading
import time
import traceback
import urllib.request
import warnings

warnings.filterwarnings("ignore")
faulthandler.enable(file=sys.__stderr__, all_threads=True)  # a hard crash (segfault, abort) prints where it happened
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")

WAN2GP_DIR = os.path.abspath("Wan2GP")
WORK_DIR = os.path.dirname(WAN2GP_DIR)
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
MODELS_DIR = os.environ.get("H3_MODELS_DIR", "/kaggle/tmp/h3_models")
LORA_DIR = os.environ.get("H3_LORA_DIR", "/kaggle/tmp/h3_loras")
VARIANT = os.environ.get("H3_VARIANT", "pdmd")
DIT_FILE = os.environ.get("H3_DIT_FILE", "MiniMax-H3-FL2VA-pruned_rank8_int8_convrot.safetensors")
TE_FILE = os.environ.get("H3_TE_FILE", "qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf")
TURBO_LORA_FILE = os.environ.get("H3_TURBO_LORA", "lora_model_0.safetensors" if VARIANT == "pdmd" else "")  # distillation LoRA
PIN_TEXT_ENCODER = os.environ.get("H3_PIN_TE", "1") == "1"
SERVER_PORT = 7860
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

import numpy as np
import psutil
import torch

import gradio as gr


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _ram_in_use():
    """total - available: unlike psutil's 'used', this counts pinned (page-locked) memory, which Linux files as shared."""
    vm = psutil.virtual_memory()
    return f"{_gb(vm.total - vm.available)} / {_gb(vm.total)}"


def _fmt_secs(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


class ConsoleBar:
    """One live progress line per phase, redrawn every second with a carriage return (Kaggle's cell output overwrites it in place).
    A finished phase is replaced by a one-line summary. Step-based phases (denoising) interpolate inside the current
    step from the previous step's duration, so the bar keeps moving and the ETA counts down between callbacks."""

    WIDTH, LINE = 24, 104
    WATCHED = {"decode": 120, "stitch": 120, "audio": 120, "save": 120}  # phase -> seconds before a stack dump

    def __init__(self, stream):
        self.out = stream
        self.lock = threading.RLock()
        self.phase = None
        self.visible = False
        self.text_pending = False  # another writer is in the middle of a line: don't draw over it
        threading.Thread(target=self._loop, daemon=True).start()

    def start(self, key, label, total=None, unit=""):
        with self.lock:
            if self.phase is not None and self.phase["key"] == key:
                return
            self.finish()
            now = time.time()
            self.phase = dict(key=key, label=label, total=total, unit=unit, done=0, t0=now, t_done=now, dt=None)
            if key in self.WATCHED:  # a phase that should take seconds: dump every thread's stack if it hangs
                faulthandler.dump_traceback_later(self.WATCHED[key], file=sys.__stderr__)
            self._draw()

    def update(self, done, total=None):
        with self.lock:
            p = self.phase
            if p is None:
                return
            now = time.time()
            if total:
                p["total"] = total
            if done > p["done"]:
                p["dt"] = (now - p["t_done"]) / (done - p["done"])
                p["t_done"], p["done"] = now, done
            self._draw()

    def finish(self):
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            elapsed = time.time() - p["t0"]
            line = f"  ✅ {p['label']}"
            if p["total"]:
                line += f" {p['done']}/{p['total']} {p['unit']}"
            line += f" in {_fmt_secs(elapsed)}"
            if p["unit"] == "steps" and p["done"]:
                line += f" ({elapsed / p['done']:.1f}s/step)"
            self._clear()
            self.out.write(line + "\n")
            self.out.flush()

    def abort(self):
        """End the live line of a phase that failed (no ✅ summary)."""
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            self._clear()
            self.out.write(f"  ❌ {p['label']} failed after {_fmt_secs(time.time() - p['t0'])}\n")
            self.out.flush()

    def _render(self):
        p, now = self.phase, time.time()
        elapsed = now - p["t0"]
        if not p["total"]:
            spinner = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"[int(now * 4) % 10]
            return f"  {spinner} {p['label']} | {_fmt_secs(elapsed)}"
        progress = p["done"]
        if p["dt"] and p["done"] < p["total"]:
            progress += min(0.97, (now - p["t_done"]) / p["dt"])
        frac = min(1.0, progress / p["total"])
        fill = int(round(frac * self.WIDTH))
        line = (f"  ⏳ {p['label']:<16} [{'█' * fill}{'░' * (self.WIDTH - fill)}] {frac * 100:3.0f}% | "
                f"{p['done']}/{p['total']} {p['unit']}")
        if p["dt"]:
            eta = max(0.0, (p["total"] - p["done"]) * p["dt"] - (now - p["t_done"]))
            line += f" | {p['dt']:.1f}s each | ETA {_fmt_secs(eta)}"
        return line + f" | {_fmt_secs(elapsed)}"

    def _draw(self):
        if self.phase is None or self.text_pending:
            return
        self.out.write("\r" + self._render().ljust(self.LINE))
        self.out.flush()
        self.visible = True

    def _clear(self):
        if self.visible:
            self.out.write("\r" + " " * self.LINE + "\r")
            self.visible = False

    def _loop(self):
        while True:
            time.sleep(1.0)
            with self.lock:
                self._draw()


class _BarAwareStream:
    """stdout / stderr wrapper: ordinary log lines clear the live bar first and print above it."""

    def __init__(self, stream, bar):
        self._stream, self._bar = stream, bar

    def write(self, data):
        with self._bar.lock:
            if data:
                self._bar._clear()
                self._bar.text_pending = not data.endswith("\n")
            return self._stream.write(data)

    def flush(self):
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


BAR = ConsoleBar(sys.stdout)
sys.stdout = _BarAwareStream(sys.stdout, BAR)
sys.stderr = _BarAwareStream(sys.stderr, BAR)


if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise SystemExit("❌ This notebook needs Kaggle's 'GPU T4 x2' accelerator (Settings → Accelerator → GPU T4 x2).")

DIT_DEV = torch.device("cuda", 0)   # first half of the DiT blocks
MAIN_DEV = torch.device("cuda", 1)  # second half + text encoder + VAEs + sampler state
torch.cuda.set_device(MAIN_DEV)     # Wan2GP / MMGP use the current device for "cuda"

torch.backends.cuda.enable_flash_sdp(False)       # Turing (SM 7.5) has no flash kernel
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cudnn.benchmark = False

_props = torch.cuda.get_device_properties(0)
print("=" * 70)
print("🎬 MiniMax H3 Turbo INT8 - Video + Audio Generator | AIQUEST Academy")
print(f"⚡ GPUs: 2 x {_props.name} ({_gb(_props.total_memory)} each, SM {_props.major}.{_props.minor}) | "
      f"RAM: {_gb(psutil.virtual_memory().total)} ({_gb(psutil.virtual_memory().available)} free)")
print("=" * 70)
sys.stdout.flush()

# Turing has no BF16 convolution kernels. Convolutions that reach BF16 (the Qwen3-VL patch embedding) run in FP32.
if _props.major < 8:
    def _conv_fp32_on_turing(original):
        def forward(self, input, weight, bias):
            if input.is_cuda and input.dtype == torch.bfloat16:
                bias32 = None if bias is None else bias.float()
                return original(self, input.float(), weight.float(), bias32).to(torch.bfloat16)
            return original(self, input, weight, bias)
        return forward
    for _conv in (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Conv3d):
        _conv._conv_forward = _conv_fp32_on_turing(_conv._conv_forward)

# ==== Wan2GP runtime: quantization handlers + INT8 kernels (what wgp.py does at import) ====
from mmgp import offload, quant_router

quant_router.unregister_handler(".fp8_quanto_bridge")
for _handler in ("shared.qtypes.int8_convrot", "shared.qtypes.gguf"):  # DiT + VAE decoder, Qwen3-VL GGUF
    quant_router.register_handler(_handler)
from shared.qtypes import gguf as _gguf_handler

quant_router.register_file_extension("gguf", _gguf_handler)

from shared.kernels import int8_backend

try:
    int8_backend.configure("auto", 1)
except Exception as e:
    print(f"⚠️ INT8 kernels unavailable, using the PyTorch path: {e}")
# Kitchen splits each INT8 GEMM into row tiles that fit a 16 MB scratch buffer, which on a T4 means 64-row GEMMs
# for the 28672-wide FFN. 128 MB gives ~700-row tiles: far better tensor-core use for ~100 MB of extra scratch.
int8_backend._SCRATCH_BYTES = 128 * 1024 * 1024
INT8_LABEL = {"kitchen": "Comfy Kitchen INT8 tensor cores", "triton": "Triton INT8 kernels"}.get(
    int8_backend._backend, "PyTorch INT8 fallback (slow)")
offload.shared_state["_attention"] = "sdpa"

from shared.utils import files_locator as fl

fl.set_checkpoints_paths([MODELS_DIR, LORA_DIR, "ckpts", "."])

import models.minimax_h3.minimax_h3_main as h3_main
from models.minimax_h3 import pipeline as h3_pipeline
from models.minimax_h3.minimax_h3_handler import family_handler
from models.minimax_h3.minimax_h3_main import VIDEO_VAE_INT8_FILE

# Two-phase (low-res + upscale) generation is not used: skip loading the 0.7 GB latent upscaler.
h3_main._load_latent_upscaler = lambda *args, **kwargs: None

# ComfyUI-native pruned checkpoints also carry the AdaLN basis / mean they were compressed with. Wan2GP's model has
# no such tensors (it derives LoRA conversions from its own canonical maps), so drop them before loading.
_orig_strip_wrappers = h3_main._strip_wrappers


def _strip_wrappers(state_dict, *args, **kwargs):
    for key in [k for k in state_dict if k.endswith(("adaln_basis", "adaln_mean"))]:
        state_dict.pop(key)
    return _orig_strip_wrappers(state_dict, *args, **kwargs)


h3_main._strip_wrappers = _strip_wrappers

# ==== Text encoder compute dtype ====
# Without llama.cpp kernels (no wheel for Kaggle's Python), each GGUF linear is dequantized to a dense weight and run
# as a normal matmul. Turing has no BF16 tensor cores, so a BF16 GEMM can be slower than plain FP32. Measure both on
# the real layer shape and keep the faster one (FP32 is also the more precise of the two).
def _gemm_seconds(dtype, m=512, k=5120, n=25600, reps=3):
    a = torch.randn(m, k, device=MAIN_DEV, dtype=dtype)
    b = torch.randn(n, k, device=MAIN_DEV, dtype=dtype)
    torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    t = time.time()
    for _ in range(reps):
        torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    return (time.time() - t) / reps


try:
    _bf16_s, _fp32_s = _gemm_seconds(torch.bfloat16), _gemm_seconds(torch.float32)
    TE_DTYPE = torch.float32 if _fp32_s < _bf16_s else torch.bfloat16
    print(f"📏 Text encoder GEMM on {_props.name}: BF16 {_bf16_s * 1000:.0f} ms, FP32 {_fp32_s * 1000:.0f} ms "
          f"-> using {'FP32' if TE_DTYPE == torch.float32 else 'BF16'}")
except Exception as e:
    TE_DTYPE = torch.bfloat16
    print(f"⚠️ GEMM benchmark failed ({e}), text encoder stays in BF16")
torch.cuda.empty_cache()
_orig_load_text_encoder = h3_main._load_text_encoder
h3_main._load_text_encoder = lambda filename, dtype: _orig_load_text_encoder(filename, TE_DTYPE)

# ==== Load the model ====
VARIANTS = {  # label, Wan2GP model type, Reference → Video support, default sampler, default steps
    "pdmd": ("PDMD 4-step (base H3 + PDMD LoRA)", "minimax_h3_fl2va_pruned", False, "euler", 4),
    "dmad": ("DMAD 4-step (base H3 + DMAD LoRA)", "minimax_h3_fl2va_pruned", False, "euler", 4),
    "fused": ("MATLOWAI fused turbo (Ref2VA)", "minimax_h3_ref2va_pruned", True, "res_multistep", 4),  # Ref2VA AdaLN table (verified)
    "official": ("Ref2VA + lightx2v 8-step LoRA", "minimax_h3_ref2va_pruned", True, "res_multistep", 8),
}
VARIANT_LABEL, MODEL_TYPE, REFERENCES_SUPPORTED, DEFAULT_SAMPLER, DEFAULT_STEPS = VARIANTS[VARIANT]
print(f"\n📦 Loading MiniMax H3 {VARIANT_LABEL} INT8 ConvRot + Qwen3-VL-32B GGUF...")
sys.stdout.flush()
_t_load = time.time()
BAR.start("load", "Reading checkpoints")
model_def = family_handler.query_model_def(MODEL_TYPE, {})
model_def["qkv_splitting"] = False             # keep the fused QKV GEMM: bigger tiles, and LoRAs map 1:1
model_def["video_vae_file"] = VIDEO_VAE_INT8_FILE  # INT8 ConvRot VAE decoder: half the VRAM, Kitchen GEMMs
pipeline, _ = family_handler.load_model(
    model_filename=[os.path.join(MODELS_DIR, DIT_FILE)],
    model_type=MODEL_TYPE,
    base_model_type=MODEL_TYPE,
    model_def=model_def,
    dtype=torch.bfloat16,
    VAE_dtype=torch.float32,
    text_encoder_filename=os.path.join(MODELS_DIR, "Qwen3-VL-32B-Instruct", TE_FILE),
)
transformer = pipeline.transformer
transformer.cache = None

# Modules H3 pins to FP32 (patch projections, output heads, AdaLN curve projections) are converted here,
# since this transformer is not handed to MMGP's profile (which would otherwise apply the locks).
for _module in transformer.modules():
    _lock = getattr(_module, "_lock_dtype", None)
    if _lock is not None and isinstance(_module, torch.nn.Linear) and _module.weight.dtype.is_floating_point:
        _module.to(_lock)

# ==== Tensor-parallel DiT: each block is split in half across the two T4s ====
# Within a block, GPU 0 computes attention heads 0-27 and FFN columns 0-7167, GPU 1 the other half. Both start from
# the same residual stream (kept as a replica on each GPU), and after attention and after the FFN each GPU sends its
# partial output projection to the other, so both GPUs hold the full sum. The weight slices follow the ConvRot
# layout exactly: QKV / FFN-up are split by output rows (Q, K, V are grouped), output / FFN-down by input columns on
# 256-wide rotation-group boundaries, so every INT8 GEMM computes precisely its share of the full product.
from models.minimax_h3.transformer import MLP as H3MLP, Attention as H3Attention, _gated_residual, _norm_modulate, _take
from shared.qtypes.int8_convrot import _rotate_activation

DEVICES = (DIT_DEV, MAIN_DEV)
_tp_cache = {"rope": None, "rope_copy": None, "temb": None, "temb_copy": None, "src": None, "replica": None}


class Int8Shard(torch.nn.Module):
    """One GPU's slice of an INT8 ConvRot linear (a set of output rows, or a range of input columns)."""

    def __init__(self, linear, device, rows=None, cols=None):
        super().__init__()
        data, scale = linear.weight._data, linear.weight._scale.reshape(-1)
        if rows is not None:
            data, scale = data.index_select(0, rows), scale.index_select(0, rows)
        if cols is not None:
            data = data[:, cols]
        # copy=True: own storage even on the same device (never a view into the memory-mapped checkpoint)
        self.register_buffer("qdata", data.contiguous().to(device, copy=True), persistent=False)
        self.register_buffer("qscale", scale.float().contiguous().to(device, copy=True), persistent=False)
        self.group = int(getattr(linear, "_convrot_group_size", 0) or 0)
        self.out_features, self.in_features = self.qdata.shape

    def forward(self, x):
        x2 = x.reshape(-1, self.in_features)
        if int8_backend.kitchen_enabled() and self.group == 256 and x2.is_cuda:
            out = int8_backend._linear_impl(x2, self.qdata, self.qscale, None, True)
        else:  # reference path (no Kitchen): rotate, dequantize, matmul
            weight = self.qdata.to(x2.dtype) * self.qscale.to(x2.dtype)[:, None]
            out = torch.nn.functional.linear(_rotate_activation(x2, self.group), weight)
        return out.reshape(*x.shape[:-1], self.out_features)


def _copy_to(value, dev, slot):
    """Per-forward copies of temb / rope for GPU 0 (they are the same tensor object for all 50 blocks)."""
    if value is None:
        return None
    first = value[0] if isinstance(value, tuple) else value
    if first.device == dev:
        return value
    if _tp_cache[slot] is not first:
        _tp_cache[slot] = first
        _tp_cache[slot + "_copy"] = tuple(v.to(dev) for v in value) if isinstance(value, tuple) else value.to(dev)
    return _tp_cache[slot + "_copy"]


def _all_reduce_residual(hiddens, partials, gates, segments):
    """hidden += gate * (partial_0 + partial_1) on both GPUs."""
    incoming = [partials[1].to(DEVICES[0], non_blocking=True, copy=True),
                partials[0].to(DEVICES[1], non_blocking=True, copy=True)]
    out = []
    for s, dev in enumerate(DEVICES):
        with torch.cuda.device(dev):
            branch = partials[s].add_(incoming[s])
            out.append(_gated_residual([hiddens[s]], [gates[s]], [branch], segments))
    return out


# ==== Long-video mode: the same block math on 4096-row slices ====
# A normal block builds several full-length intermediates per GPU (modulated input, fused QKV, BF16 and FP16 copies of
# Q/K/V, the output-projection input, the exchanged partial sums): roughly 75-110 KB per token. For long or 720p clips
# that does not fit next to half of the model, so in this mode only what attention needs at full length is kept (the
# residual stream and FP16 Q/K/V, ~40 KB per token). Norms, projections, RoPE, the FFN and the GPU exchange run on
# 8192-row slices (~0.4 GB of temporaries). Same arithmetic, a little more Python per block; attention dominates long clips either way.
import models.minimax_h3.transformer as _h3_transformer

LONG_VIDEO = {"on": False, "rows": 8192}


def _row_chunks(n):
    step = LONG_VIDEO["rows"]
    return [(a, min(n, a + step)) for a in range(0, n, step)]


def _slice_segments(segments, a, b):
    """(start, stop, modality row) segments of the packed sequence, clipped to rows [a, b) and shifted to 0."""
    return [(max(s0, a) - a, min(s1, b) - a, row) for s0, s1, row in segments if min(s1, b) > max(s0, a)]


def _apply_rope(query, key, rope):
    """H3's rotary embedding on [1, n, heads, dim] tensors, in place (Attention.forward's unfused path)."""
    pairs = rope.shape[-2]
    cosine, sine = rope[..., 0], rope[..., 1]
    scratch = torch.empty_like(query[..., :pairs])
    for tensor in (query, key):
        first, second = tensor[..., :pairs], tensor[..., pairs:2 * pairs]
        scratch.copy_(first)
        first.mul_(cosine).addcmul_(second, sine, value=-1)
        second.mul_(cosine).addcmul_(scratch, sine)


def _sliced_attention(shard, hidden, mod, segments, rope):
    attn = shard.attn
    rows, heads, dim = hidden.shape[0], attn.heads, attn.head_dim
    inner = heads * dim
    if isinstance(rope, tuple):
        rope = rope[0]
    q, k, v = (torch.empty(1, rows, heads, dim, device=hidden.device, dtype=torch.float16) for _ in range(3))
    for a, b in _row_chunks(rows):
        h = _norm_modulate(shard.norm1, hidden[a:b], [mod[0], mod[1]], _slice_segments(segments, a, b))
        qkv = attn.qkv_proj(h)
        del h
        query, key, value = (t.view(1, b - a, heads, dim) for t in qkv.split(inner, dim=-1))
        query, key = attn.q_norm(query), attn.k_norm(key)
        if rope is not None:
            _apply_rope(query, key, rope[:, a:b])
        q[:, a:b].copy_(query)
        k[:, a:b].copy_(key)
        v[:, a:b].copy_(value)
        del qkv, query, key, value
    qkv_list = [q, k, v]
    del q, k, v
    out = _h3_transformer.pay_attention(qkv_list, recycle_q=True)[0]  # [rows, heads, dim] FP16
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(rows):
        partial[a:b] = attn.out_proj(out[a:b].reshape(b - a, inner).to(hidden.dtype))
    return partial


def _sliced_mlp(shard, hidden, mod, segments):
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(hidden.shape[0]):
        h = _norm_modulate(shard.norm2, hidden[a:b], [mod[3], mod[4]], _slice_segments(segments, a, b))
        partial[a:b] = shard.mlp([h])
    return partial


def _all_reduce_sliced(hiddens, partials, gates, segments):
    """_all_reduce_residual without full-length copies: the partial sums cross between GPUs slice by slice."""
    for a, b in _row_chunks(hiddens[0].shape[0]):
        incoming = [partials[1][a:b].to(DEVICES[0], non_blocking=True, copy=True),
                    partials[0][a:b].to(DEVICES[1], non_blocking=True, copy=True)]
        for s, dev in enumerate(DEVICES):
            with torch.cuda.device(dev):
                partials[s][a:b].add_(incoming[s])
        del incoming
    out = []
    for s, dev in enumerate(DEVICES):
        with torch.cuda.device(dev):
            out.append(_gated_residual([hiddens[s]], [gates[s]], [partials[s]], segments))
    return out


class TPBlock(torch.nn.Module):
    def __init__(self, block):
        super().__init__()
        attn, mlp = block.attn, block.mlp
        heads, head_dim = attn.heads, attn.head_dim
        inner, ffn, hidden = heads * head_dim, mlp.ffn, mlp.hidden
        half_inner, half_ffn = inner // 2, ffn // 2
        self.splits = []
        shards = []
        for s, dev in enumerate(DEVICES):
            h = torch.arange(s * half_inner, (s + 1) * half_inner)
            f = torch.arange(s * half_ffn, (s + 1) * half_ffn)
            qkv_rows = torch.cat((h, h + inner, h + 2 * inner))
            fc1_rows = torch.cat((f, f + ffn))
            out_cols = slice(s * half_inner, (s + 1) * half_inner)
            fc2_cols = slice(s * half_ffn, (s + 1) * half_ffn)
            self.splits.append({"qkv": qkv_rows, "fc1": fc1_rows, "out": out_cols, "fc2": fc2_cols})
            with torch.device("meta"):
                sattn = H3Attention(hidden, heads // 2, head_dim, attn.q_norm.eps, dtype=torch.bfloat16)
                smlp = H3MLP(hidden, half_ffn, mlp.chunk_size, dtype=torch.bfloat16)
            sattn.qkv_proj = Int8Shard(attn.qkv_proj, dev, rows=qkv_rows)
            sattn.out_proj = Int8Shard(attn.out_proj, dev, cols=out_cols)
            sattn.q_norm = copy.deepcopy(attn.q_norm).to(dev)
            sattn.k_norm = copy.deepcopy(attn.k_norm).to(dev)
            smlp.fc1 = Int8Shard(mlp.fc1, dev, rows=fc1_rows)
            smlp.fc2 = Int8Shard(mlp.fc2, dev, cols=fc2_cols)
            shard = torch.nn.Module()
            shard.attn, shard.mlp = sattn, smlp
            shard.norm1 = copy.deepcopy(block.norm1).to(dev)
            shard.norm2 = copy.deepcopy(block.norm2).to(dev)
            shard.adaln_proj = copy.deepcopy(block.adaln_proj).to(dev)
            shards.append(shard)
        self.shards = torch.nn.ModuleList(shards)
        self.dims = {"qkv": (hidden, 3 * inner), "out": (inner, hidden), "fc1": (hidden, 2 * ffn), "fc2": (ffn, hidden)}

    @property
    def attn(self):  # H3's forward checks block.attn for q_proj / vdn
        return self.shards[1].attn

    def forward(self, x_list, temb, segments, rope, residual_signature_elements=0):
        if residual_signature_elements:
            raise RuntimeError("First Block Cache is not supported by the tensor-parallel DiT")
        hidden = _take(x_list)
        hidden = hidden if hidden.device == MAIN_DEV else hidden.to(MAIN_DEV)
        replica = _tp_cache["replica"] if _tp_cache["src"] is hidden else hidden.to(DIT_DEV, copy=True)
        hiddens = [replica, hidden]
        tembs = [_copy_to(temb, DIT_DEV, "temb"), temb]
        ropes = [_copy_to(rope, DIT_DEV, "rope"), rope]
        if LONG_VIDEO["on"]:
            return self._forward_sliced(hiddens, tembs, ropes, segments)
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            shard = self.shards[s]
            with torch.cuda.device(dev):
                mod = shard.adaln_proj(tembs[s])
                mods.append(mod)
                h = _norm_modulate(shard.norm1, hiddens[s], [mod[0], mod[1]], segments)
                partials.append(shard.attn([h], rope=ropes[s]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            shard = self.shards[s]
            with torch.cuda.device(dev):
                h = _norm_modulate(shard.norm2, hiddens[s], [mods[s][3], mods[s][4]], segments)
                partials.append(shard.mlp([h]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]

    def _forward_sliced(self, hiddens, tembs, ropes, segments):
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            with torch.cuda.device(dev):
                mods.append(self.shards[s].adaln_proj(tembs[s]))
                partials.append(_sliced_attention(self.shards[s], hiddens[s], mods[s], segments, ropes[s]))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            with torch.cuda.device(dev):
                partials.append(_sliced_mlp(self.shards[s], hiddens[s], mods[s], segments))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]


NUM_BLOCKS = len(transformer.blocks)
BAR.start("split", "Splitting DiT", NUM_BLOCKS, "blocks")
for _i in range(NUM_BLOCKS):
    transformer.blocks[_i] = TPBlock(transformer.blocks[_i])
    BAR.update(_i + 1)
BAR.finish()
for _name, _child in transformer.named_children():
    if _name not in ("blocks", "token_refiner"):
        _child.to(MAIN_DEV)
for _name, _buffer in list(transformer.named_buffers(recurse=False)):
    transformer._buffers[_name] = _buffer.to(MAIN_DEV)
gc.collect()

# ==== Exact-size pinned RAM ====
# PyTorch's pinned-memory allocator rounds every block up to the next power of two: MMGP's ~210 MB text-encoder
# chunks become 256 MB each (+2.8 GB) and the token refiner's 17 tensors would take 2.25 GB for 1.44 GB. Page-locking
# ordinary RAM with cudaHostRegister pins exactly the bytes asked for (same fast async copies to the GPU).
_HOST_REGISTERED = []  # registered RAM stays alive (and pinned) for the whole session
PINNED_BYTES = {}


def _dont_fork(ptr, nbytes):
    """madvise(MADV_DONTFORK): a fork() must not duplicate pinned RAM. Linux cannot share pinned pages copy-on-write,
    so it copies them into the child right away: forking this process (subprocess with preexec_fn does) would need
    another ~13 GB at once and gets the engine killed. CUDA marks its own cudaHostAlloc blocks this way."""
    try:
        import ctypes
        page = os.sysconf("SC_PAGE_SIZE")
        start = -(-ptr // page) * page
        end = (ptr + nbytes) // page * page
        if end > start:
            ctypes.CDLL("libc.so.6").madvise(ctypes.c_void_p(start), ctypes.c_size_t(end - start), 10)  # MADV_DONTFORK
    except Exception:
        pass


def pinned_empty_exact(nbytes):
    t = torch.empty(nbytes, dtype=torch.uint8)
    torch.cuda.check_error(torch.cuda.cudart().cudaHostRegister(t.data_ptr(), nbytes, 0))
    _dont_fork(t.data_ptr(), nbytes)
    _HOST_REGISTERED.append(t)
    return t


_torch_empty = torch.empty


def _empty_exact_pinned(*size, pin_memory=False, **kwargs):
    """torch.empty for MMGP's pinning: its non-power-of-two uint8 chunks are registered at their exact size."""
    n = size[0] if len(size) == 1 and isinstance(size[0], int) else 0
    if pin_memory and n > 0 and n & (n - 1) and kwargs.get("dtype") == torch.uint8:
        try:
            return pinned_empty_exact(n)
        except Exception:
            pass
    return _torch_empty(*size, pin_memory=pin_memory, **kwargs)


# The 1.4 GiB token refiner runs once per prompt: keep it in pinned RAM and swap it onto GPU 1 only for that.
_refiner = transformer.token_refiner
_ref_list = list(_refiner.parameters())
_ref_offsets, _ref_total = [], 0
for _p in _ref_list:
    _ref_offsets.append(_ref_total)
    _ref_total += -(-_p.numel() * _p.element_size() // 64) * 64
try:
    _ref_buf = pinned_empty_exact(_ref_total)
    _refiner_params = []
    for _p, _off in zip(_ref_list, _ref_offsets):
        _cpu = _ref_buf[_off:_off + _p.numel() * _p.element_size()].view(_p.dtype).view(_p.shape)
        _cpu.copy_(_p.data)
        _refiner_params.append((_p, _cpu))
except Exception as e:  # no cudaHostRegister: PyTorch's (rounded) pinned blocks
    print(f"⚠️ Exact pinning unavailable ({e}); using PyTorch pinned memory")
    _refiner_params = [(p, p.data.to("cpu").pin_memory()) for p in _ref_list]
PINNED_BYTES["token refiner"] = _ref_total
for _p, _cpu in _refiner_params:
    _p.data = _cpu
_orig_preprocess_text = transformer.preprocess_text_embeds


def _preprocess_text_embeds(text_states):
    if text_states.shape[-1] == transformer.hidden_size:
        return text_states
    for p, cpu in _refiner_params:
        p.data = cpu.to(MAIN_DEV, non_blocking=True)
    try:
        return _orig_preprocess_text(text_states.to(MAIN_DEV))
    finally:
        for p, cpu in _refiner_params:
            p.data = cpu


transformer.preprocess_text_embeds = _preprocess_text_embeds

# ==== Text encoder + VAEs: MMGP on GPU 1, unloaded before denoising ====
pipeline.audio_vae._convertWeightsFloatTo = None  # keep the audio VAE in FP32 (Turing has no BF16 convs)
aux_pipe = {
    "text_encoder": pipeline.text_encoder.language_model,
    "vision_encoder": pipeline.text_encoder.visual,
    "vae": pipeline.video_decoder,
    "video_encoder": pipeline.video_encoder,
    "audio_vae": pipeline.audio_vae,
}
pipeline.video_decoder._budget = 0  # decode with the whole VAE resident (it is called once per clip and tile)
if TE_DTYPE == torch.float32:
    pipeline.text_encoder.language_model._convertWeightsFloatTo = None
    pipeline.text_encoder.visual._convertWeightsFloatTo = None
# The text encoder (13.6 GB) and the video VAE decoder (2.3 GB) are pinned in RAM. Kaggle's disk reads at ~180 MB/s,
# so a memory-mapped copy that has dropped out of the page cache costs ~15 s per video before decoding can even
# start; from pinned RAM the same weights reach the GPU in well under a second. The small audio VAE and image
# encoder stay as they are (a few seconds at most).
_pin = (["text_encoder"] if PIN_TEXT_ENCODER else []) + ["vae"]
BAR.start("pin", "Pinning text encoder + video decoder in RAM (one-time)")
try:
    torch.empty = _empty_exact_pinned
    try:
        aux_offload = offload.profile(aux_pipe, profile_no=4, quantizeTransformer=False, convertWeightsFloatTo=torch.bfloat16,
                                      pinnedMemory=_pin, asyncTransfers=True, verboseLevel=-1,
                                      budgets={"text_encoder": 1500, "vision_encoder": 0, "*": 3000})
    finally:
        torch.empty = _torch_empty
except Exception as e:
    if not _pin:
        raise
    print(f"⚠️ Pinning failed ({e}); text encoder streams from regular RAM")
    aux_offload = offload.profile(aux_pipe, profile_no=4, quantizeTransformer=False, convertWeightsFloatTo=torch.bfloat16,
                                  pinnedMemory=False, asyncTransfers=True, verboseLevel=-1,
                                  budgets={"text_encoder": 1500, "vision_encoder": 0, "*": 3000})
pipeline.set_offload_handoff(aux_offload, None)  # pipeline unloads these before handing the GPU to the DiT
BAR.finish()
for _key, _label in (("text_encoder", "text encoder"), ("vae", "video decoder")):
    PINNED_BYTES[_label] = getattr(aux_pipe[_key], "_pinned_bytes", 0)
if hasattr(torch._C, "_host_emptyCache"):
    torch._C._host_emptyCache()  # MMGP's temporary probe block

gc.collect()
for _dev in (DIT_DEV, MAIN_DEV):
    with torch.cuda.device(_dev):
        torch.cuda.empty_cache()
_free0, _free1 = torch.cuda.mem_get_info(0)[0], torch.cuda.mem_get_info(1)[0]
print(f"✅ Loaded in {time.time() - _t_load:.0f}s | {INT8_LABEL}")
print(f"   GPU 0: half of all {NUM_BLOCKS} DiT blocks ({_gb(torch.cuda.memory_allocated(0))}), {_gb(_free0)} free")
print(f"   GPU 1: other half + heads ({_gb(torch.cuda.memory_allocated(1))}), {_gb(_free1)} free")
print(f"   RAM in use: {_ram_in_use()} | pinned " + " + ".join(f"{k} {_gb(v)}" for k, v in PINNED_BYTES.items() if v))
try:
    _a = torch.empty(64 * 2**20, dtype=torch.uint8, device=DIT_DEV)
    _b = torch.empty(64 * 2**20, dtype=torch.uint8, device=MAIN_DEV)
    _a.to(MAIN_DEV); _b.to(DIT_DEV); torch.cuda.synchronize(0); torch.cuda.synchronize(1)
    _t = time.time()
    for _ in range(4):
        _a.to(MAIN_DEV); _b.to(DIT_DEV)
    torch.cuda.synchronize(0); torch.cuda.synchronize(1)
    print(f"   GPU<->GPU exchange: {8 * 64 / 1024 / (time.time() - _t):.1f} GB/s "
          f"(peer access: {torch.cuda.can_device_access_peer(0, 1)})")
    del _a, _b
except Exception as e:
    print(f"   GPU<->GPU exchange benchmark skipped: {e}")
sys.stdout.flush()

# ==== LoRA runtime ====
# Applied as forward hooks on the resident (two-GPU) transformer: out += strength * (x @ A^T) @ B^T, written in place
# with addmm_ so no output-sized temporary is created. Keys go through Wan2GP's own H3 converter, which maps
# diffusers / kohya names and re-projects AdaLN adapters onto the pruned 8-wide AdaLN curves.
LORA_CATALOG = {
    "🧑 Realism People (fal)": {
        "repo": "fal/MiniMax-H3-Realism-People-LoRA", "file": "h3-realism-people-t2v-i2v-r2v.safetensors",
        "scale": 1.0, "trigger": "r34l1sm", "info": "Natural skin, faces, expressions and film lighting for people shots."},
    "🎥 Camera Motion": {
        "repo": "Jojocodex/minimax-h3-Camera-Motion-lora", "file": "camera_motion_h3_lora_v1_3000_pruned.safetensors",
        "scale": 0.9, "trigger": "camera motion", "info": "Push-in, pull-back, orbit, aerial, handheld... describe the move after the trigger."},
    "🔄 360° Orbit (same image as first + last frame)": {
        "repo": "pablodawson/MiniMax-H3-360-Orbit-LoRA", "file": "minimax_h3_flf2v_lora_v1.safetensors",
        "scale": 1.0, "trigger": "One frozen instant. Only the camera moves. In a continuous 360 orbit.",
        "info": "Put the SAME image as start and end frame: the camera orbits the frozen scene and lands back on it."},
    "⚔️ Weapon Combat (BUNNY)": {
        "repo": "JOKER141/MiniMax-H3-Weapon-Combat-LoRA", "file": "Bunny_weapon_combatV1.safetensors",
        "scale": 1.0, "trigger": "BUNNY", "info": "Weapon swings, blocks, parries and dynamic fight choreography."},
    "🧱 Spatial & Physics": {
        "repo": "Jojocodex/minimax-h3-spatial-physics-lora", "file": "wushu_spatial_physics_clean_3000_pruned.safetensors",
        "scale": 0.4, "trigger": "", "info": "More physical collisions, bounces and gravity. Keep it at 0.3-0.5."},
    "🗣️ Natural Face & Speech": {
        "repo": "vpakarinen/natural-face-speech-h3-lora", "file": "natural_face_speech_h3_lora_v2_500.safetensors",
        "scale": 0.7, "trigger": "", "info": "Talking heads: facial muscle motion and clear speech. Put the line in 'Spoken line'."},
    "🎞️ Studio 1939 Animation": {
        "repo": "lovis93/studio-1939-old-animation-lora-minimax-h3", "file": "studio1939-light.safetensors",
        "scale": 1.0, "trigger": "gulliv3r,", "info": "Hand-painted golden-age animation look."},
    "🌸 Anime Motion": {
        "repo": "prithivMLmods/MiniMax-H3-I2V-Anime-Motion-LoRA", "file": "MiniMax-H3-I2V-Anime-Motion-LoRA-1400.safetensors",
        "scale": 1.0, "trigger": "Anime-Motion", "info": "Subtle looping anime motion, best with a start image."},
    "📼 VHS Tape (1980s)": {
        "repo": "KennethFal/vh5tape-vhs-lora-minimax-h3", "file": "vh5tape.safetensors",
        "scale": 1.0, "trigger": "vh5tape, worn VHS tape recording with visible tape damage.",
        "info": "Worn VHS look and sound. Damage tiers: lightly worn / worn / badly damaged."},
}
NO_LORA = "None"
_LORA_PREFIXES = ("model.diffusion_model.", "diffusion_model.", "base_model.model.", "transformer.", "model.")
_LORA_SUFFIXES = {".default.weight": ".weight", ".lora.A.weight": ".lora_A.weight", ".lora.B.weight": ".lora_B.weight",
                  ".lora.down.weight": ".lora_down.weight", ".lora.up.weight": ".lora_up.weight"}


LORA_ROWS = 8192


class LoraRuntime:
    """LoRAs as forward hooks. Each original layer name maps to one or more target modules (the two halves of a
    tensor-parallel layer), and every target gets the matching slice of the LoRA: B rows for row-split layers, A columns
    for column-split layers (whose partial outputs are summed across GPUs, so a bias delta goes to one half only)."""

    def __init__(self, model, model_type):
        self.model = model
        self.model_type = model_type
        self.targets = {}  # name -> (in_features, out_features, [(module, device, dtype, kind, index)])
        for name, module in model.named_modules():
            if isinstance(module, torch.nn.Linear) and not name.startswith("blocks."):
                dev = MAIN_DEV if name.startswith("token_refiner") or not module.weight.is_cuda else module.weight.device
                dtype = torch.float32 if module.weight.dtype == torch.float32 else torch.bfloat16
                self.targets[name] = (module.in_features, module.out_features, [(module, dev, dtype, None, None)])
        for i, block in enumerate(model.blocks):
            for short, attr, kind in (("qkv", "attn.qkv_proj", "rows"), ("out", "attn.out_proj", "cols"),
                                      ("fc1", "mlp.fc1", "rows"), ("fc2", "mlp.fc2", "cols")):
                parts = [(functools.reduce(getattr, attr.split("."), block.shards[s]), DEVICES[s], torch.bfloat16, kind,
                          block.splits[s][short]) for s in range(2)]
                self.targets[f"blocks.{i}.{attr}"] = (*block.dims[short], parts)
            adaln = [(block.shards[s].adaln_proj.linear, DEVICES[s], torch.float32, None, None) for s in range(2)]
            first = adaln[0][0]
            self.targets[f"blocks.{i}.adaln_proj.linear"] = (first.in_features, first.out_features, adaln)
        self.loaded = {}    # key -> [(module, [(A, B, out_start, out_end)], alpha_scale, diff_b)]
        self.strength = {}  # key -> current strength
        self.active = {}    # id(module) -> [(key, segments, alpha_scale, diff_b)]
        self.handles = {}   # id(module) -> hook handle

    def _normalize(self, state_dict):
        out = {}
        for key, value in state_dict.items():
            for prefix in _LORA_PREFIXES:
                if key.startswith(prefix):
                    key = key[len(prefix):]
                    break
            for src, dst in _LORA_SUFFIXES.items():
                if key.endswith(src):
                    key = key[:-len(src)] + dst
                    break
            out[key] = value
        return self.model.preprocess_loras(self.model_type, out)

    def _iter_groups(self, path):
        """(layer name, {"A", "B", "alpha", "diff_b"}) for every layer of a LoRA file, read one block at a time: a
        rank-128 LoRA is 1.4 GB, and Wan2GP's converter makes several copies of what it is given, which a fully loaded
        file does not leave room for next to the pinned text encoder."""
        from safetensors import safe_open

        with safe_open(path, framework="pt", device="cpu") as f:
            batches = {}
            for k in f.keys():
                m = re.search(r"blocks[._](\d+)[._]", k)
                batches.setdefault(("refiner" if "refiner" in k else "block", int(m.group(1))) if m else ("other", 0), []).append(k)
            for keys in batches.values():
                groups = {}
                for k, v in self._normalize({k: f.get_tensor(k) for k in keys}).items():
                    for down, up in (("lora_A.weight", "lora_B.weight"), ("lora_down.weight", "lora_up.weight")):
                        if k.endswith("." + down):
                            groups.setdefault(k[:-len(down) - 1], {})["A"] = v
                        elif k.endswith("." + up):
                            groups.setdefault(k[:-len(up) - 1], {})["B"] = v
                    if k.endswith(".alpha"):
                        groups.setdefault(k[:-len(".alpha")], {})["alpha"] = float(v.item() if torch.is_tensor(v) else v)
                    elif k.endswith(".diff_b"):
                        groups.setdefault(k[:-len(".diff_b")], {})["diff_b"] = v
                yield from groups.items()

    def _read_groups(self, path):
        return dict(self._iter_groups(path))

    @staticmethod
    def _segments(A, B):
        """[(A_i, B_i, out_start, out_end)]. Separate q / k / v adapters arrive fused as cat(A) and block_diag(B),
        which is 2/3 zeros: they are split back into their three blocks (a third of the memory and the work)."""
        r3, o3 = A.shape[0], B.shape[0]
        if r3 % 3 == 0 and o3 % 3 == 0 and r3 >= 3:
            r, o = r3 // 3, o3 // 3
            off = [B[i * o:(i + 1) * o, j * r:(j + 1) * r] for i in range(3) for j in range(3) if i != j]
            if all(not bool(t.any()) for t in off):
                return [(A[i * r:(i + 1) * r], B[i * o:(i + 1) * o, i * r:(i + 1) * r], i * o, (i + 1) * o) for i in range(3)]
        return [(A, B, 0, o3)]

    def load(self, key, path):
        adapters, layers, skipped = [], 0, 0
        for name, g in self._iter_groups(path):
            target = self.targets.get(name)
            A, B, diff_b = g.get("A"), g.get("B"), g.get("diff_b")
            if target is None or (A is None) != (B is None) or (A is None and diff_b is None):
                skipped += 1
                continue
            in_f, out_f, parts = target
            if A is not None and (A.shape[1] != in_f or B.shape[0] != out_f or A.shape[0] != B.shape[1]):
                skipped += 1
                continue
            alpha_scale = g["alpha"] / A.shape[0] if A is not None and "alpha" in g else 1.0
            segments = self._segments(A, B) if A is not None else []
            for n, (module, dev, dtype, kind, index) in enumerate(parts):
                segs, d = [], diff_b
                if dtype == torch.bfloat16:  # BF16 layers get their LoRA math in FP16 on the tensor cores (see _hook)
                    dtype = torch.float16
                for a, b, o0, o1 in segments:
                    if kind == "rows":  # this shard computes the output rows `index` (in order)
                        pos = ((index >= o0) & (index < o1)).nonzero().flatten()
                        if not len(pos):
                            continue
                        assert int(pos[-1] - pos[0]) + 1 == len(pos), "non-contiguous LoRA rows in a shard"
                        b, o0, o1 = b.index_select(0, index[pos] - o0), int(pos[0]), int(pos[-1]) + 1
                    elif kind == "cols":
                        a = a[:, index]
                    segs.append((a.to(dev, dtype).contiguous(), b.to(dev, dtype).contiguous(), o0, o1))
                if kind == "rows" and diff_b is not None:
                    d = diff_b.index_select(0, index)
                elif kind == "cols":
                    d = diff_b if n == 0 else None
                adapters.append((module, segs, alpha_scale, None if d is None else d.to(dev, dtype)))
            layers += 1
        if not adapters:
            raise ValueError(f"No layer of this LoRA matches MiniMax H3 ({skipped} groups checked)")
        self.loaded[key] = adapters
        mb = sum(sum(t.numel() * t.element_size() for seg in segs for t in seg[:2]) for _, segs, _, _ in adapters) / 2**20
        print(f"  🎨 LoRA '{key}': {layers} layers ({mb:.0f} MB VRAM){f', {skipped} unmatched groups skipped' if skipped else ''}")

    def _hook(self, module_id):
        def hook(module, args, output):
            adapters = self.active.get(module_id)
            if not adapters:
                return output
            x = args[0]
            x2 = x.reshape(-1, x.shape[-1])
            out2 = output.view(-1, output.shape[-1])
            for key, segments, alpha_scale, diff_b in adapters:
                s = self.strength.get(key, 0.0) * alpha_scale
                if s == 0.0:
                    continue
                if segments and segments[0][0].dtype == torch.float16:
                    # FP16 tensor cores: a rank-128 LoRA on every layer is ~3% of the FLOPs, but in FP32 (no tensor
                    # cores on Turing, and BF16 is slower still) it cost ~30% of each step. The input is scaled by a
                    # power of two before the FP16 cast and the result scaled back, so large activations (FFN-down
                    # input) cannot overflow; FP16's 11-bit mantissa is plenty for a correction term.
                    pre = 1.0 / 512 if x2.shape[-1] > 8192 else 1.0 / 64
                    for start in range(0, x2.shape[0], LORA_ROWS):
                        end = min(x2.shape[0], start + LORA_ROWS)
                        xs = (x2[start:end] * pre).to(torch.float16)
                        for a16, b16, o0, o1 in segments:
                            delta = (xs @ a16.t()) @ b16.t()
                            out2[start:end, o0:o1].add_(delta.to(out2.dtype), alpha=s / pre)
                            del delta
                        del xs
                elif segments:  # FP32 layers (AdaLN curves)
                    for start in range(0, x2.shape[0], LORA_ROWS):
                        end = min(x2.shape[0], start + LORA_ROWS)
                        xf = x2[start:end].float()
                        for a32, b32, o0, o1 in segments:
                            out2[start:end, o0:o1].add_(((xf @ a32.float().t()) @ b32.float().t()).mul_(s).to(out2.dtype))
                        del xf
                if diff_b is not None:
                    out2.add_(diff_b, alpha=s)
            return output
        return hook

    def apply(self, selections):
        """selections: {key: (path, strength)} - loads new LoRAs, frees unused ones, sets strengths."""
        for key in [k for k in self.loaded if k not in selections]:
            del self.loaded[key]
        for key, (path, strength) in selections.items():
            if key not in self.loaded:
                self.load(key, path)
            self.strength[key] = float(strength)
        self.active, modules = {}, {}
        for key, adapters in self.loaded.items():
            for module, segments, alpha_scale, diff_b in adapters:
                self.active.setdefault(id(module), []).append((key, segments, alpha_scale, diff_b))
                modules[id(module)] = module
        for module_id in list(self.handles):
            if module_id not in self.active:
                self.handles.pop(module_id).remove()
        for module_id, module in modules.items():
            if module_id not in self.handles:
                self.handles[module_id] = module.register_forward_hook(self._hook(module_id))
        gc.collect()
        for dev in DEVICES:
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()


lora_runtime = LoraRuntime(transformer, MODEL_TYPE)

# ==== DiT attention backend: PyTorch SDPA or tensor-core GEMM attention, whichever is faster on this GPU ====
# Attention is most of the DiT's work (it grows with the square of the video length: a 10 s clip costs ~3.5x a 5 s
# one). On Turing, PyTorch's memory-efficient SDPA kernel runs well below what cuBLAS FP16 GEMMs reach, so an exact
# chunked attention (Q·Kᵀ GEMM -> softmax -> P·V GEMM, a few hundred query rows at a time) can be faster. Both are
# timed on the real head shape at startup; GEMM is used only if it is faster and matches SDPA. The VAE keeps
# Wan2GP's own attention and 256 px tiles untouched.
import shared.attention as _w2g_attention

_orig_pay_attention = _w2g_attention.pay_attention
ATTENTION = {"dit": "sdpa"}
ATTENTION_CHUNK_BYTES = 640 * 2**20  # logits + probabilities of one chunk


def gemm_attention(q, k, v):
    """Exact softmax attention for [B, L, H, D] FP16 tensors using batched GEMMs over query-row chunks."""
    batch, length, heads, dim = q.shape
    keys = k.shape[1]
    rows = max(64, min(length, ATTENTION_CHUNK_BYTES // (4 * heads * keys)))
    out = torch.empty_like(q)
    q = q * (dim ** -0.5)
    for b in range(batch):
        q_b, k_t, v_b = q[b].transpose(0, 1), k[b].permute(1, 2, 0), v[b].transpose(0, 1)  # strided views, no copies
        for start in range(0, length, rows):
            probs = torch.softmax(torch.matmul(q_b[:, start:start + rows], k_t), dim=-1)
            out[b, start:start + rows] = torch.matmul(probs, v_b).transpose(0, 1)
            del probs
    return out


def _fast_pay_attention(kind):
    def pay_attention(qkv_list, *args, **kwargs):
        if (ATTENTION[kind] != "gemm" or args or kwargs.get("attention_mask") is not None or kwargs.get("causal")
                or kwargs.get("q_lens") is not None or kwargs.get("k_lens") is not None):
            return _orig_pay_attention(qkv_list, *args, **kwargs)
        q, k, v = qkv_list
        qkv_list.clear()
        out_dtype = q.dtype
        q, k, v = [t if t.dtype == torch.float16 else t.to(torch.float16) for t in (q, k, v)]
        return gemm_attention(q, k, v).to(out_dtype)
    return pay_attention


_h3_transformer.pay_attention = _fast_pay_attention("dit")


def _time_it(fn, reps=2):
    fn()
    torch.cuda.synchronize()
    t = time.time()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return (time.time() - t) / reps


def benchmark_attention(kind, batch, length, heads, dim):
    try:
        with torch.cuda.device(DIT_DEV):
            q, k, v = (torch.randn(batch, length, heads, dim, device=DIT_DEV, dtype=torch.float16) for _ in range(3))
            sdpa = lambda: _orig_pay_attention([q, k, v], force_attention="sdpa")
            gemm = lambda: gemm_attention(q, k, v)
            ref, got = sdpa().float(), gemm().float()
            err = ((got - ref).norm() / ref.norm()).item()
            t_sdpa, t_gemm = _time_it(sdpa), _time_it(gemm)
            del q, k, v, ref, got
            torch.cuda.empty_cache()
        ATTENTION[kind] = "gemm" if t_gemm < 0.95 * t_sdpa and err < 1e-2 else "sdpa"
        print(f"📏 DiT attention ({length} tokens x {heads} heads): SDPA {t_sdpa * 1000:.0f} ms, "
              f"GEMM {t_gemm * 1000:.0f} ms (diff {err:.1e}) -> using {ATTENTION[kind].upper()}")
    except Exception as e:
        ATTENTION[kind] = "sdpa"
        print(f"⚠️ {kind} attention benchmark failed ({e}); using SDPA")


benchmark_attention("dit", 1, 16384, 28, 128)  # 480p / 5 s, half the heads per GPU

# ==== VAE decode on both GPUs ====
# The video VAE decodes the latent in independent temporal clips (only the cross-fade between neighbours is
# sequential), so the clips are split between the two GPUs: GPU 1 uses the MMGP-managed decoder, GPU 0 a second
# decoder whose weights are copied, only for the duration of the decode, from the first one (no second copy in RAM;
# between videos it holds no weights at all). Tiling, attention and cross-fades are exactly Wan2GP's. On the first video GPU 0 also
# decodes clip 0 and must match GPU 1's result; otherwise decoding falls back to GPU 1 alone.
_vae_b = h3_main._load_video_vae(VIDEO_VAE_INT8_FILE, False)  # same architecture; its own weights are dropped below
_vae_b.encoder = _vae_b.quant_conv = None  # decode-only copy
_vae_b_parts = torch.nn.ModuleDict({"post_quant_conv": _vae_b.post_quant_conv, "decoder": _vae_b.decoder})
_vae_b_slots = []  # (module, "_parameters" | "_buffers", name, source module)
_source_modules = dict(pipeline.video_decoder.named_modules())
for _name, _module in _vae_b_parts.named_modules():
    for _store in ("_parameters", "_buffers"):
        for _key, _value in getattr(_module, _store).items():
            if _value is not None:
                _vae_b_slots.append((_module, _store, _key, _source_modules[_name]))


def _vae_b_attach(device):
    """Copy the first decoder's weights (pinned RAM, or GPU 1 if MMGP has it loaded) to `device`."""
    for module, store, key, source in _vae_b_slots:
        value = getattr(source, store)[key].to(device, non_blocking=True)
        if store == "_parameters":
            value = torch.nn.Parameter(value, requires_grad=False)
        getattr(module, store)[key] = value


def _vae_b_release():
    """Drop every weight of the second decoder. It must not keep references to the first decoder's tensors either:
    MMGP swaps in new GPU tensors when it loads a model, so a shared reference taken while the decoder is on GPU 1
    would keep those 2.3 GB of VRAM alive after MMGP unloads it."""
    for module, store, key, _ in _vae_b_slots:
        getattr(module, store)[key] = None


_vae_b_release()
gc.collect()

# Wan2GP decodes each clip as 256 px tiles stacked into ONE decoder batch: ~12 tiles at 480p (decode already peaks at
# ~13.3 GB of the T4's 14.6), ~28 at 720p, which does not fit next to half of the DiT + the LoRA. The decoder is a ViT
# that treats every tile on its own, so running the tiles in smaller groups gives exactly the same frames. The group
# size is planned per GPU from its free VRAM before each decode (~80 MB per tile measured, 100 MB budgeted), so the
# decoder never has to run out of memory; halving on an out-of-memory error stays only as a last resort.
VAE_TILE_GROUP = {}
VAE_TILE_BYTES = 100 * 2**20


def _tile_grouped(decoder):
    full_forward = decoder.forward

    def forward(hidden_states, *args, **kwargs):
        dev, tiles = hidden_states.device.index, hidden_states.shape[0]
        while True:
            n = min(VAE_TILE_GROUP.get(dev, tiles), tiles)
            try:
                if n >= tiles:
                    return full_forward(hidden_states, *args, **kwargs)
                return torch.cat([full_forward(hidden_states[i:i + n], *args, **kwargs) for i in range(0, tiles, n)])
            except torch.OutOfMemoryError:
                if n <= 1:
                    raise
                VAE_TILE_GROUP[dev] = max(1, n // 2)
                print(f"  ⚙️ VAE: GPU {dev} decodes {VAE_TILE_GROUP[dev]} tiles at a time from now on (out of memory with {n})")
                with torch.cuda.device(hidden_states.device):
                    torch.cuda.synchronize()  # kernels queued before the failed allocation may still use freed blocks
                    gc.collect()
                    torch.cuda.empty_cache()

    decoder.forward = forward


_tile_grouped(pipeline.vae.decoder)
_tile_grouped(_vae_b.decoder)


def _plan_tile_groups():
    """Tiles per decoder call that fit each GPU's free VRAM right now (GPU 1 loads its 2.3 GB decoder on first use)."""
    resident = any(p.is_cuda for p in pipeline.video_decoder.parameters())
    for dev in DEVICES:
        with torch.cuda.device(dev):
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        free = torch.cuda.mem_get_info(dev)[0]
        if dev == MAIN_DEV and not resident:
            free -= PINNED_BYTES.get("video decoder", 0)
        VAE_TILE_GROUP[dev.index] = max(1, int((free - 512 * 2**20) // VAE_TILE_BYTES))


# ==== Drop the checkpoint files from the page cache ====
# Loading reads ~40 GB of checkpoints, and Linux keeps whatever fits of them cached in RAM. That cache is free to
# reclaim (it never causes an out-of-memory restart), but Kaggle's RAM meter counts it, so the meter shows several GB
# more than the engine really uses. Everything is already pinned or on the GPUs, so the cached copies are useless.
# Pages a model still has memory-mapped are left alone by the kernel.
def drop_file_cache(*roots):
    for root in roots:
        for dirpath, _, files in os.walk(root, followlinks=True):
            for name in files:
                try:
                    fd = os.open(os.path.join(dirpath, name), os.O_RDONLY)
                    try:
                        os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                    finally:
                        os.close(fd)
                except (OSError, AttributeError):
                    pass


def _ram_report():
    vm = psutil.virtual_memory()
    cache = getattr(vm, "cached", 0) + getattr(vm, "buffers", 0)
    return f"{_ram_in_use()} used by the engine | {_gb(cache)} file cache (reclaimable, Kaggle's meter counts it)"


drop_file_cache(MODELS_DIR)
print(f"   RAM: {_ram_report()}")
_orig_vae_decode = pipeline.vae._decode
PARALLEL_VAE = {"enabled": True, "verified": False}
PEAKS = {}


def parallel_vae_decode(z):
    vae = pipeline.vae
    chunk_tokens, token_drop = vae.tokens_chunk_size, vae.config.token_drop
    ratio = vae.temporal_compression_ratio
    chunk_frames = chunk_tokens * ratio
    if z.shape[2] == 1:
        return _orig_vae_decode(z)
    num_tokens = z.shape[2] + token_drop
    pad_tokens = (-num_tokens) % chunk_tokens
    num_chunks = (num_tokens + pad_tokens) // chunk_tokens - int(token_drop > 0)
    if num_chunks < 2 or not PARALLEL_VAE["enabled"]:
        return _orig_vae_decode(z)
    if pad_tokens > 0:
        z = torch.cat([z, z[:, :, -1:].repeat(1, 1, pad_tokens, 1, 1)], dim=2)
    intra_tail = vae.config.clip_length % ratio
    tokens_before_pad = z.shape[2] - pad_tokens
    pad_frames = sum(intra_tail if intra_tail and (tokens_before_pad + i) % chunk_tokens == 0 else ratio
                     for i in range(pad_tokens))
    output_frames = num_chunks * (chunk_frames - vae.frame_pre_padding) + vae.frame_overlap - pad_frames

    _vae_b.use_tiling = vae.use_tiling
    for attr in ("tile_sample_min_height", "tile_sample_min_width", "tile_sample_min_overlap_height", "tile_sample_min_overlap_width"):
        setattr(_vae_b, attr, getattr(vae, attr))
    PEAKS["denoise"] = [torch.cuda.max_memory_allocated(d) for d in DEVICES]  # the DiT phase ends here
    for d in DEVICES:
        torch.cuda.reset_peak_memory_stats(d)
    clips, check, errors, done, lock = [None] * num_chunks, {}, [], [0], threading.Lock()
    BAR.start("decode", "VAE decoding", num_chunks, "clips")

    def decode_one(dev, model, i):
        start = i * chunk_tokens
        clip = model._decode_clip(z[:, :, start:start + chunk_tokens + vae.token_overlap].to(dev))
        return clip.to("cpu")

    def work(dev, model, indices):
        try:
            with torch.cuda.device(dev), torch.inference_mode():
                for i in indices:
                    if pipeline._interrupt:
                        return
                    if i < 0:  # verification copy of clip 0 on GPU 0
                        check["clip"] = decode_one(dev, model, 0)
                        continue
                    clips[i] = decode_one(dev, model, i)
                    with lock:
                        done[0] += 1
                        BAR.update(done[0])
                        report(0.86 + 0.11 * done[0] / num_chunks, f"🎞️ VAE decoding {done[0]}/{num_chunks} clips")
        except BaseException as e:
            errors.append(e)

    with torch.cuda.device(DIT_DEV):
        torch.cuda.empty_cache()
        _vae_b_attach(DIT_DEV)
    _plan_tile_groups()
    gpu0_clips = list(range(1, num_chunks, 2))
    if not PARALLEL_VAE["verified"]:
        gpu0_clips = [-1] + gpu0_clips
    try:
        helper = threading.Thread(target=work, args=(DIT_DEV, _vae_b, gpu0_clips))
        helper.start()
        work(MAIN_DEV, vae, list(range(0, num_chunks, 2)))
        helper.join()
    finally:
        try:  # let every queued kernel finish before GPU 0's decoder weights are dropped and its memory released
            for dev in DEVICES:
                torch.cuda.synchronize(dev)
        except Exception as e:
            errors.append(e)
        _vae_b_release()  # drop GPU 0's copy
        if not errors:
            with torch.cuda.device(DIT_DEV):
                torch.cuda.empty_cache()
    if errors:
        raise errors[0]
    if pipeline._interrupt:
        from models.minimax_h3.interrupt import GenerationInterrupted
        raise GenerationInterrupted
    if not PARALLEL_VAE["verified"] and "clip" in check:
        a, b = check["clip"].float(), clips[0].float()
        mismatch = ((a - b).norm() / b.norm().clamp_min(1e-6)).item()
        PARALLEL_VAE["verified"] = True
        if mismatch > 1e-2:
            PARALLEL_VAE["enabled"] = False
            print(f"⚠️ GPU 0's VAE copy disagrees with GPU 1 (rel diff {mismatch:.3f}); decoding on GPU 1 only from now on")
            with torch.cuda.device(MAIN_DEV), torch.inference_mode():
                for i in range(1, num_chunks, 2):
                    clips[i] = decode_one(MAIN_DEV, vae, i)
        else:
            print(f"  ✅ Two-GPU VAE decode verified (GPU 0 vs GPU 1 rel diff {mismatch:.1e})")

    BAR.start("stitch", "Stitching clips")
    decoded, write_position, overlap = None, 0, None
    for i in range(num_chunks):
        clip = clips[i].float()
        clips[i] = None
        for j in range(int(token_drop > 0) + 1):
            chunk = clip[:, :, j * chunk_frames:(j + 1) * chunk_frames][:, :, vae.frame_pre_padding:]
            if j == 0:
                if overlap is not None:
                    chunk = vae._blend(overlap, chunk, vae.frame_overlap, dim=-3)
                copy_frames = min(chunk.shape[2], output_frames - write_position)
                if copy_frames > 0:
                    chunk = vae._prepare_decoded_chunk(chunk[:, :, :copy_frames])
                    if decoded is None:
                        # FP16 halves the RAM of a long 720p video (3.9 -> 2 GB); [-1, 1] keeps 0.06-level precision of 8 bits
                        decoded = torch.empty(*chunk.shape[:2], output_frames, *chunk.shape[3:], dtype=torch.float16)
                    decoded[:, :, write_position:write_position + copy_frames].copy_(chunk)
                    write_position += copy_frames
            else:
                overlap = chunk.contiguous()
    if overlap is not None and write_position < output_frames:
        copy_frames = min(overlap.shape[2], output_frames - write_position)
        chunk = vae._prepare_decoded_chunk(overlap[:, :, :copy_frames])
        decoded[:, :, write_position:write_position + copy_frames].copy_(chunk)
        write_position += copy_frames
    if write_position != output_frames:
        raise RuntimeError(f"MiniMax H3 VAE decoded {write_position} frames, expected {output_frames}")
    return decoded


pipeline.vae._decode = parallel_vae_decode
_orig_audio_decode = pipeline.audio_vae.decode


def _audio_decode(*args, **kwargs):
    BAR.start("audio", "Audio decoding")
    return _orig_audio_decode(*args, **kwargs)


pipeline.audio_vae.decode = _audio_decode


# The distillation LoRA is applied live to every generation, exactly as trained. Merging it into the INT8 weights
# does not work: its changes are ~0.1-0.2% of the weights, smaller than half an INT8 step for >99% of them, so
# re-quantizing rounds almost all of it away (measured on the real FL2VA weights: 0.1-25% of the effect survived),
# which leaves the undistilled base model sampled in 4 steps (soft, painterly video). Cost: ~0.85 GB of VRAM per GPU
# and a few % of each step (FP16 tensor cores; it was ~30% in FP32).
PERMANENT_LORAS = {}
if TURBO_LORA_FILE:
    _distill_label = {"pdmd": "⚡ PDMD 4-step", "dmad": "⚡ DMAD 4-step"}.get(VARIANT, "⚡ Turbo (lightx2v)")
    PERMANENT_LORAS[_distill_label] = (os.path.join(LORA_DIR, TURBO_LORA_FILE), 1.0)
    lora_runtime.apply(PERMANENT_LORAS)
    drop_file_cache(LORA_DIR)
    print(f"   RAM after the LoRA: {_ram_report()} | GPU free: "
          f"{_gb(torch.cuda.mem_get_info(0)[0])} / {_gb(torch.cuda.mem_get_info(1)[0])}")
    sys.stdout.flush()


def fetch_lora(entry_or_url):
    """Download a catalog LoRA or a direct .safetensors URL (Hugging Face 'resolve' links work) into LORA_DIR."""
    if isinstance(entry_or_url, dict):
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=entry_or_url["repo"], filename=entry_or_url["file"], local_dir=LORA_DIR)
    url = entry_or_url.strip()
    m = re.match(r"https?://huggingface\.co/([^/]+/[^/]+)/(?:resolve|blob)/[^/]+/(.+?)(?:\?.*)?$", url)
    if m:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=m.group(1), filename=urllib.request.unquote(m.group(2)), local_dir=LORA_DIR)
    name = re.sub(r"[^A-Za-z0-9._-]", "_", os.path.basename(url.split("?")[0])) or "custom_lora.safetensors"
    if not name.endswith(".safetensors"):
        name += ".safetensors"
    path = os.path.join(LORA_DIR, name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
    return path


# ==== Prompt builder (MiniMax H3 prompt contract) ====
LANGUAGES = ["English", "Chinese", "Spanish", "French", "German", "Japanese", "Korean", "Hindi", "Urdu", "Arabic",
             "Portuguese", "Russian", "Italian", "Turkish", "Indonesian"]
_STRUCTURED = re.compile(r"(integrated_multimodal_description:|detailed_description:|subject_definitions:)")


def build_prompt(scene, dialogue, language, sound, music, duration_s, has_start, has_end, n_refs, ref_names, has_voice,
                 triggers):
    scene = (scene or "").strip()
    triggers = [t.strip() for t in triggers if t and t.strip().lower() not in scene.lower()]
    if _STRUCTURED.search(scene):  # the user wrote a full H3 prompt: keep it, only put trigger words in front
        return (" ".join(triggers) + " " + scene).strip() if triggers else scene
    # LoRA trigger words open the shot description, the way the LoRAs were captioned
    trigger = "".join(t + (" " if t[-1] in ".,;:" else ", ") for t in triggers)
    secs = f"{duration_s:.1f}".rstrip("0").rstrip(".")
    speaker = "<Subject 1>" if n_refs else "The main character"
    line = ""
    if dialogue and dialogue.strip():
        voice = " in the voice from <Audio 1>" if has_voice else ""
        line = (f" {speaker} speaks clearly{voice} with natural, synchronized lip movements and says "
                f"(S1) <d>[{language}] {dialogue.strip()}</d>")
    sound = (sound or "").strip() or "Natural ambient sound that matches the scene and the visible action."
    music = (music or "").strip() or "N/A"
    pictures = 0
    anchors = []
    if has_start:
        pictures += 1
        anchors.append(f"For the target video, at 0.00 seconds into the target video, <Picture {pictures}> (from [Shot 1]) is fully referenced.")
    if has_end:
        pictures += 1
        anchors.append(f"For the target video, at {duration_s:.2f} seconds into the target video, <Picture {pictures}> (from [Shot 1]) is fully referenced.")
    if not n_refs:
        lead = "Continue directly from <Picture 1>, preserving its subject, composition and lighting. " if has_start else ""
        body = (f"integrated_multimodal_description: [Shot 1] {trigger}A {secs}-second cinematic single take. {lead}{scene}{line}\n"
                f"overall_soundscape: {sound}\nnon_diegetic_music: {music}")
        return "\n".join(anchors + [body])
    names = [n.strip() for n in (ref_names or "").split(",") if n.strip()]
    subjects, retention = [], []
    for i in range(n_refs):
        label = names[i] if i < len(names) else ("the main subject" if i == 0 else "the subject")
        picture = pictures + i + 1
        subjects.append(f"<Subject {i + 1}> is {label} in <Picture {picture}>, preserving their exact identity, "
                        f"appearance, proportions, clothing and distinctive details.")
        retention.append(f"<Subject {i + 1}> (appears in [Shot 1]): fully_preserved - identity and appearance are retained; "
                         f"the setting, action, camera and lighting are new.")
    if has_voice:
        retention.append("<Audio 1>: reference - the voice timbre and delivery of <Subject 1> follow <Audio 1>.")
    task = "[reference generation + audio reference]" if has_voice else "[reference generation]"
    return "\n".join(anchors + [
        "subject_definitions:", *subjects,
        "summary:", f"{task} {scene}",
        "retention_analysis:", *retention,
        "detailed_description:",
        f"{trigger}The target video is a {secs}-second cinematic single take. [Shot 1] {scene}{line} Identities, motion and "
        f"lighting remain stable and physically coherent.",
        f"overall_soundscape: {sound}",
        f"non_diegetic_music: {music}",
    ])


# ==== Sizes, durations and a VRAM guard ====
RESOLUTIONS = {"360p (fastest)": (640, 352), "480p (recommended)": (832, 480), "540p": (960, 544), "720p (slow)": (1280, 704)}
ASPECTS = {"Auto (match image)": None, "16:9 Landscape": 16 / 9, "9:16 Portrait": 9 / 16, "1:1 Square": 1.0,
           "4:3 Classic": 4 / 3, "3:4 Portrait": 3 / 4, "21:9 Cinema": 21 / 9}
STEP_PRESETS = {"⚡ Fast - 4 steps": 4, "⚖️ Balanced - 6 steps": 6, "💎 Quality - 8 steps": 8}
DURATIONS = {"3 s": 73, "4.5 s": 107, "5 s": 124, "6.5 s": 158, "8 s": 192, "10 s": 243, "12 s": 294, "15 s": 362}


def size_for(resolution, aspect, image):
    w0, h0 = RESOLUTIONS[resolution]
    ratio = ASPECTS.get(aspect)
    if ratio is None:
        ratio = image.width / image.height if image is not None else 16 / 9
    area = w0 * h0
    w = max(256, round(math.sqrt(area * ratio) / 32) * 32)
    h = max(256, round(area / w / 32) * 32)
    return w, h


def estimate_rows(w, h, frames, images_in_prompt):
    per_frame = (h // 32) * (w // 32)
    video = h3_pipeline.video_latent_frames(frames) * per_frame
    audio = round(frames / 24 * 40) * 2
    text = 1200 + images_in_prompt * per_frame  # prompt tokens + Qwen vision tokens of each picture
    return video + audio + images_in_prompt * per_frame + text


def free_for_denoise():
    aux_offload.unload_all()
    gc.collect()
    frees = []
    for dev in DEVICES:
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
        frees.append(torch.cuda.mem_get_info(dev)[0])
    return frees


def needed_bytes(rows, long_video=False):
    """Working VRAM per GPU: activations per packed row (half the heads per GPU) + scratch. Estimates."""
    return rows * (48 if long_video else 110) * 1024 + 700 * 1024**2


# ==== Live progress (also polled by the page on the Cloudflare link, which holds back Gradio's SSE stream) ====
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0, "finished": 0.0, "video": None, "message": ""}


def report(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


def _to_video_tensor(image):
    return h3_pipeline._pil_to_video(image.convert("RGB"))


def _audio_to_wav(path):
    out = os.path.join(OUTPUT_DIR, ".gradio_cache", f"voice_{int(time.time() * 1000)}.wav")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-ac", "2", "-ar", "32000", out], check=True)
    import soundfile as sf
    return out, sf.info(out).duration


@torch.inference_mode()  # the decoded video is an inference tensor
def save_mp4(video, audio, sample_rate, path, fps=24):
    """Streams [3, T, H, W] video in [-1, 1] to ffmpeg a few frames at a time: no full-size copies in RAM."""
    import soundfile as sf

    _, t, h, w = video.shape
    wav = path[:-4] + ".wav"
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{w}x{h}", "-r", str(fps), "-i", "-"]
    if audio is not None:
        sf.write(wav, np.asarray(audio, dtype=np.float32), int(sample_rate))
        cmd += ["-i", wav, "-c:a", "aac", "-b:a", "192k"]
    cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "17", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
            "-shortest", path]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    try:
        for start in range(0, t, 16):
            frames = video[:, start:start + 16].to(torch.float32, copy=True).clamp_(-1, 1).add_(1).mul_(127.5).round_()
            proc.stdin.write(frames.to(torch.uint8).permute(1, 2, 3, 0).contiguous().numpy().tobytes())
        proc.stdin.close()
    except BrokenPipeError:
        pass
    if proc.wait() != 0:
        raise RuntimeError(f"ffmpeg failed with exit code {proc.returncode}")
    if os.path.exists(wav):
        os.remove(wav)


@torch.inference_mode()
def run_generation(prompt_text, frames, w, h, steps, sampler, shift, seed, start_img=None, end_img=None, ref_imgs=(),
                   voice_path=None, progress=None):
    pipeline._interrupt = False
    t0 = time.time()
    timings = {"text": 0.0, "denoise": 0.0, "decode": 0.0}
    phase = {"name": "prep", "t": time.time(), "step_t": time.time()}

    def ui(fraction, text):
        report(fraction, text)
        if progress is not None:
            try:
                progress(fraction, desc=text)
            except Exception:
                pass

    def callback(step, latent=None, is_start=False, override_num_inference_steps=None, progress_unit=None,
                 progress_title=None, **kwargs):
        now = time.time()
        if progress_unit == "layers":  # Qwen3-VL text / image encoding
            if phase["name"] != "text":
                phase.update(name="text", t=now)
            total = override_num_inference_steps or 50
            BAR.start("text", "Prompt encoding", total, "layers")
            BAR.update(max(0, step + 1), total)
            ui(0.02 + 0.13 * min(1.0, (step + 1) / total), f"📝 {progress_title or 'Encoding prompt'} {max(0, step + 1)}/{total}")
            return
        if progress_unit == "tiles":  # VAE encode / decode
            if "Decod" in (progress_title or "") and phase["name"] != "decode":
                if phase["name"] == "denoise":
                    timings["denoise"] = now - phase["t"]
                phase.update(name="decode", t=now)
            total = override_num_inference_steps or 1
            decoding = "Decod" in (progress_title or "")
            BAR.start("decode" if decoding else "encode", "VAE decoding" if decoding else "Image encoding", total, "tiles")
            BAR.update(max(0, step + 1), total)
            ui(0.86 + 0.11 * min(1.0, (step + 1) / total), f"🎞️ {progress_title or 'VAE'} {max(0, step + 1)}/{total}")
            return
        if is_start:
            if phase["name"] == "text":
                timings["text"] = now - phase["t"]
            phase.update(name="denoise", t=now, step_t=now, total=override_num_inference_steps or steps)
            BAR.start("denoise", "Denoising", phase["total"], "steps")
            ui(0.15, f"🎬 Denoising step 0/{phase['total']}")
            return
        total = phase.get("total", steps)
        dt = now - phase["step_t"]
        phase["step_t"] = now
        done = step + 1
        BAR.update(done, total)
        ui(0.15 + 0.70 * done / total, f"🎬 Denoising step {done}/{total} ({dt:.1f}s/it)")

    kwargs = dict(input_prompt=prompt_text, frame_num=frames, height=h, width=w, sampling_steps=int(steps), seed=int(seed),
                  shift=float(shift), sample_solver=sampler, fps=24, guide_phases=1, callback=callback,
                  image_refs_relative_size=100, video_prompt_type="I" if ref_imgs else "", audio_prompt_type="")
    if start_img is not None:
        kwargs["image_start"] = _to_video_tensor(start_img)
    if end_img is not None:
        kwargs["image_end"] = _to_video_tensor(end_img)
    if ref_imgs:
        kwargs["input_ref_images"] = [img.convert("RGB") for img in ref_imgs]
    if voice_path:
        kwargs.update(audio_guide=voice_path, audio_prompt_type="A")
    ui(0.01, "⚡ Preparing...")
    BAR.start("prep", "Preparing inputs")
    try:
        result = pipeline.generate(**kwargs)
    finally:
        BAR.finish()
    if result is None:
        return None, None, None, timings
    if phase["name"] == "decode":
        timings["decode"] = time.time() - phase["t"]
    elif phase["name"] == "denoise":  # decoding reports through the parallel decoder, not the step callback
        timings["denoise"] = phase["step_t"] - phase["t"]
        timings["decode"] = time.time() - phase["step_t"]
    timings["total"] = time.time() - t0
    return result["x"], result.get("audio"), result.get("audio_sampling_rate", 32000), timings


# ==== Gradio handler ====
def on_main_gpu(fn):
    """CUDA's current device is per thread: Gradio workers start on GPU 0, which is full of DiT blocks.
    Run every handler with GPU 1 current, the device the sampler, text encoder and VAEs use for "cuda"."""
    @functools.wraps(fn)
    def wrapped(*args, **kwargs):
        with torch.cuda.device(MAIN_DEV):
            return fn(*args, **kwargs)
    return wrapped


@on_main_gpu
def generate_video(prompt, dialogue, language, sound, music, start_img, end_img, ref1, ref2, ref3, ref_names, voice,
                   resolution, aspect, duration, steps, sampler, shift, seed, lora1, s1, lora2, s2, custom_url,
                   custom_file, s3, auto_trigger, custom_steps=0, progress=None):
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    try:
        refs = [r for r in (ref1, ref2, ref3) if r is not None]
        if refs and not REFERENCES_SUPPORTED:
            return None, ("❌ Reference → Video needs the Ref2VA model: set H3_VARIANT = \"fused\" in Step 1. "
                          "This variant does text, start-frame and first + last frame to video.")
        if not (prompt or "").strip() and not refs and start_img is None:
            return None, "❌ Please write a prompt."
        frames = DURATIONS[duration]
        steps = int(custom_steps) if custom_steps and int(custom_steps) > 0 else STEP_PRESETS.get(steps, 4)
        w, h = size_for(resolution, aspect, start_img if start_img is not None else (end_img if end_img is not None else (refs[0] if refs else None)))
        if seed is None or int(seed) < 0:
            seed = random.randint(0, 2**31 - 1)
        seed = int(seed)

        # LoRAs: catalog picks + optional custom URL / file
        report(0.01, "🎨 Preparing LoRAs...")
        selections, triggers = dict(PERMANENT_LORAS), []
        for choice, strength in ((lora1, s1), (lora2, s2)):
            if choice and choice != NO_LORA and float(strength) != 0.0 and choice not in selections:
                entry = LORA_CATALOG[choice]
                selections[choice] = (fetch_lora(entry), strength)
                if auto_trigger and entry["trigger"]:
                    triggers.append(entry["trigger"])
        custom = None
        if custom_file is not None:
            custom = custom_file if isinstance(custom_file, str) else custom_file.name
        elif custom_url and custom_url.strip():
            custom = fetch_lora(custom_url)
        if custom and float(s3) != 0.0:
            selections[f"custom:{os.path.basename(custom)}"] = (custom, s3)
        lora_runtime.apply(selections)

        voice_path = None
        if voice:
            if not refs:
                return None, "❌ A voice reference needs at least one reference image (H3 binds each voice to a reference)."
            voice_path, voice_len = _audio_to_wav(voice)
            if voice_len < 2.0:
                return None, f"❌ The voice reference must be at least 2 seconds long (got {voice_len:.1f}s)."

        images_in_prompt = (start_img is not None) + (end_img is not None) + len(refs)
        rows = estimate_rows(w, h, frames, images_in_prompt)
        frees = free_for_denoise()
        LONG_VIDEO["on"] = needed_bytes(rows) > min(frees)
        if needed_bytes(rows, LONG_VIDEO["on"]) > min(frees):
            return None, (f"❌ {w}x{h} for {duration} needs about {_gb(needed_bytes(rows, True))} of working VRAM per GPU "
                          f"even in long-video mode, but only {_gb(frees[0])} (GPU 0) / {_gb(frees[1])} (GPU 1) is free. "
                          f"Pick a shorter duration or a lower resolution.")

        final_prompt = build_prompt(prompt, dialogue, language, sound, music, frames / 24, start_img is not None,
                                    end_img is not None, len(refs), ref_names, voice_path is not None, triggers)
        mode = ("Reference → Video" if refs else "First + Last Frame → Video" if start_img is not None and end_img is not None
                else "Image → Video" if start_img is not None else "Last Frame → Video" if end_img is not None else "Text → Video")
        print(f"\n{'=' * 70}\n🎬 {mode} | {w}x{h} | {frames} frames ({frames / 24:.1f}s) | {int(steps)} steps {sampler} | seed {seed}")
        if len(selections):
            print("🎨 LoRAs: " + ", ".join(f"{k} @ {float(v[1]):.2f}" for k, v in selections.items()))
        if LONG_VIDEO["on"]:
            print(f"🐢 Long-video mode: {rows} tokens, blocks run in {LONG_VIDEO['rows']}-row slices to fit the VRAM")
        print(f"📝 {final_prompt[:300]}{'...' if len(final_prompt) > 300 else ''}\n{'=' * 70}")
        sys.stdout.flush()

        for dev in DEVICES:
            torch.cuda.reset_peak_memory_stats(dev)
        PEAKS.clear()
        video, audio, sr, timings = run_generation(final_prompt, frames, w, h, steps, sampler, shift, seed, start_img, end_img,
                                                   refs, voice_path, progress=progress)
        if video is None:
            return None, "⏹️ Generation stopped."
        report(0.97, "💾 Saving MP4...")
        t_save = time.time()
        out_path = os.path.join(OUTPUT_DIR, f"h3_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed}.mp4")
        BAR.start("save", "Saving MP4")
        save_mp4(video, audio, sr, out_path)
        BAR.finish()
        total = timings.get("total", 0.0)
        per_step = timings["denoise"] / max(1, int(steps))
        status = "\n".join([
            f"✅ Done in {total:.0f}s  ({mode})",
            f"⚡ Prompt encoding {timings['text']:.0f}s | Denoising {timings['denoise']:.0f}s ({per_step:.1f}s/step) | VAE + audio {timings['decode']:.0f}s",
            f"📐 {w}x{h} | {frames} frames @ 24 fps ({frames / 24:.1f}s) | 32 kHz stereo audio",
            f"🎲 Seed {seed} | {int(steps)} steps {sampler} | shift {float(shift):g}",
            f"🎨 LoRAs: {', '.join(f'{k} @ {float(v[1]):.2f}' for k, v in selections.items()) or 'none'}",
            f"📁 {out_path}",
        ])
        print(f"✅ Saved {out_path} | total {total:.0f}s = prompt {timings['text']:.0f}s + denoise {timings['denoise']:.0f}s "
              f"({per_step:.1f}s/step) + VAE/audio {timings['decode']:.0f}s + MP4 {time.time() - t_save:.0f}s | "
              f"RAM {_ram_in_use()} | peak VRAM GPU0/GPU1: "
              + (f"denoise {_gb(PEAKS['denoise'][0])}/{_gb(PEAKS['denoise'][1])}, " if "denoise" in PEAKS else "")
              + f"decode {_gb(torch.cuda.max_memory_allocated(0))}/{_gb(torch.cuda.max_memory_allocated(1))}\n")
        drop_file_cache(LORA_DIR, OUTPUT_DIR)
        sys.stdout.flush()
        return out_path, status
    except torch.OutOfMemoryError:
        traceback.print_exc()
        return None, "❌ Out of GPU memory. Pick a lower resolution or a shorter duration and try again."
    except Exception as e:
        traceback.print_exc()
        if "CUDA error" in str(e):  # e.g. an illegal memory access: the GPU context is unusable from here on
            print("❌ The GPUs are in an error state. Re-run Step 2 to restart the engine (the models reload in ~3 min).")
            return None, f"❌ GPU error: {e}\nRe-run Step 2 to restart the engine."
        return None, f"❌ Error: {e}"
    finally:
        BAR.abort()
        PROGRESS["active"] = False
        for key in _tp_cache:  # last hidden state / rope / temb copies held on both GPUs
            _tp_cache[key] = None
        gc.collect()
        for dev in DEVICES:
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()


# ==== Generation as a background job ====
# A long clip takes 15-60 minutes, longer than the Gradio share / Cloudflare tunnels reliably keep one event stream
# open. So the Generate button only starts a job and returns; the page asks for the job's state every 2 s with short,
# independent requests (queue=False: plain HTTP, no event stream), and a reloaded page picks the result up again.
JOB = {"id": 0, "done": 0, "running": False, "video": None, "status": "Ready."}


def _job_worker(job_id, args):
    try:
        video, status = generate_video(*args)
    except BaseException as e:  # generate_video reports its own errors; this is a last resort
        video, status = None, f"❌ Error: {e}"
    JOB.update(running=False, done=job_id, video=video, status=status)
    PROGRESS.update(active=False, finished=time.time(), video=os.path.basename(video) if video else None,
                    message=status.splitlines()[0] if status else "")


def start_generation(*args):
    if JOB["running"]:
        return gr.skip(), "⏳ A video is already being generated. It appears here as soon as it is ready."
    JOB.update(id=JOB["id"] + 1, running=True)
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    threading.Thread(target=_job_worker, args=(JOB["id"], args), daemon=True).start()
    return None, ("🚀 Generation started. Progress is shown at the top of the page and the video appears here when it is "
                  "ready. Long clips take a while; reloading the page or a dropped connection does not stop the job.")


def poll_job(seen):
    """Every 2 s: live progress while a job runs, the result once (per page) when it is done."""
    if JOB["running"]:
        elapsed = time.time() - PROGRESS["started"] if PROGRESS["started"] else 0.0
        return gr.skip(), f"⏳ {PROGRESS['text'] or 'Working...'} · {_fmt_secs(elapsed)}", seen
    if JOB["done"] and JOB["done"] != seen:
        return JOB["video"], JOB["status"], JOB["done"]
    return gr.skip(), gr.skip(), seen


def stop_generation():
    if not JOB["running"]:
        return "Nothing is running."
    pipeline._interrupt = True
    return "⏹️ Stopping after the current step..."


def load_latest_video():
    videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.mp4")), key=os.path.getmtime)
    if not videos:
        return None, f"No videos in {OUTPUT_DIR} yet."
    return videos[-1], f"📁 Loaded latest video: {videos[-1]}"


def lora_info(choice):
    if not choice or choice == NO_LORA:
        return gr.update(value=1.0), ""
    entry = LORA_CATALOG[choice]
    trig = f" Trigger: `{entry['trigger']}`" if entry["trigger"] else ""
    return gr.update(value=entry["scale"]), f"ℹ️ {entry['info']}{trig}"


# ==== Gradio UI (AIQUEST Academy branding) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎬 MiniMax H3 Turbo - Video + Audio</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
STATUS_HTML = (f'<div class="status-line">⚡ {INT8_LABEL} &nbsp;|&nbsp; 20B DiT tensor-parallel on 2 x T4 '
               f'&nbsp;|&nbsp; {VARIANT_LABEL} '
               f'&nbsp;|&nbsp; 24 fps video + 32 kHz stereo audio</div>')
FOOTER_HTML = '<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved</div>'

PROGRESS_HEAD = """
<script>
(function () {
  // Progress + result overlay fed by plain HTTP polling: it keeps working when a tunnel drops Gradio's event stream.
  var box = null, fill = null, label = null, hideAt = 0;
  function build() {
    box = document.createElement("div");
    box.style.cssText = "position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:99999;width:min(560px,92vw);" +
      "background:#1f2937;border-radius:12px;box-shadow:0 8px 24px rgba(0,0,0,.25);padding:10px 14px;display:none;" +
      "font-family:Inter,sans-serif;color:#fff;font-size:13px";
    label = document.createElement("div");
    label.style.cssText = "margin-bottom:6px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis";
    var track = document.createElement("div");
    track.style.cssText = "height:8px;background:rgba(255,255,255,.18);border-radius:6px;overflow:hidden";
    fill = document.createElement("div");
    fill.style.cssText = "height:100%;width:0;background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);transition:width .6s ease";
    track.appendChild(fill); box.appendChild(label); box.appendChild(track); document.body.appendChild(box);
  }
  async function tick() {
    try {
      var r = await fetch("aiq_progress?t=" + Date.now(), {cache: "no-store"});
      var p = await r.json();
      if (!box && document.body) build();
      if (box) {
        if (p.active) {
          box.style.display = "block"; hideAt = 0;
          fill.style.width = Math.round(p.fraction * 100) + "%";
          label.textContent = "⏳ " + p.text + " · " + Math.round(p.elapsed) + "s";
        } else if (box.style.display === "block") {
          if (!hideAt) {
            fill.style.width = "100%";
            label.textContent = (p.video ? "✅ Done in " : "⚠️ Finished after ") + Math.round(p.elapsed) + "s ";
            if (p.video) {
              var link = document.createElement("a");
              link.href = "aiq_video?t=" + Date.now(); link.target = "_blank";
              link.style.cssText = "color:#c7d2fe;font-weight:600";
              link.textContent = "· ▶ open / download the video";
              label.appendChild(link);
            } else if (p.message) { label.textContent += "· " + p.message; }
            hideAt = Date.now() + 60000;
          } else if (Date.now() > hideAt) { box.style.display = "none"; }
        }
      }
    } catch (e) {}
    setTimeout(tick, 1000);
  }
  tick();
})();
</script>
"""

LORA_CHOICES = [NO_LORA] + list(LORA_CATALOG)
EXAMPLES = [
    ["A street musician plays a glowing violin on a rainy neon-lit Tokyo night, the camera slowly circles him as "
     "puddles reflect pink and blue signs and people with umbrellas pass by.",
     "", "English", "Rain on pavement, distant traffic, the violin melody echoing between buildings.", ""],
    ["A young woman in a cozy cafe looks up from her laptop, smiles warmly at the camera and leans forward.",
     "You made it! I saved you the best seat by the window.", "English",
     "Quiet cafe chatter, an espresso machine hissing, cups clinking.", "Soft lo-fi jazz."],
    ["A red fox runs across a snowy ridge at sunrise, powder spraying from every stride as a long-lens camera pans with it.",
     "", "English", "Fast paw impacts in powder, cold wind, distant crows.", "Low sustained cellos rising slowly."],
]

print("🖥️ Building the UI...")
sys.stdout.flush()
with gr.Blocks(theme=gr.themes.Soft(), css=CSS, head=PROGRESS_HEAD, title="MiniMax H3 Turbo | AIQUEST Academy") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)
    gr.Markdown("💡 **Tips:** describe subject, action, camera and light. Put exact speech in *Spoken line* for lip-synced "
                "dialogue. 480p + 5 s is the sweet spot on T4; 360p is the fastest preview. Time per step grows with the "
                "square of the length × resolution: 480p 10 s ≈ 3x and 15 s ≈ 6x a 5 s clip; 720p 15 s takes ~45 min. "
                "Long / 720p clips switch to a slower low-VRAM mode automatically.")
    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(label="🎬 Scene / action", lines=4,
                                placeholder="A cinematic shot of ... (or paste a full H3-structured prompt)")
            with gr.Row():
                dialogue = gr.Textbox(label="🗣️ Spoken line (optional)", lines=1, scale=3,
                                      placeholder="Exact words the character says")
                language = gr.Dropdown(LANGUAGES, value="English", label="🌐 Language", scale=1)
            with gr.Row():
                sound = gr.Textbox(label="🔊 Sound effects / ambience (optional)", lines=1)
                music = gr.Textbox(label="🎵 Background music (optional)", lines=1)
        with gr.Column(scale=2):
            with gr.Row():
                resolution = gr.Dropdown(list(RESOLUTIONS), value="480p (recommended)", label="📐 Resolution")
                aspect = gr.Dropdown(list(ASPECTS), value="Auto (match image)", label="📏 Aspect ratio")
            duration = gr.Dropdown(list(DURATIONS), value="5 s", label="⏱️ Duration")
            with gr.Row():
                steps = gr.Dropdown(list(STEP_PRESETS), value=next(k for k, v in STEP_PRESETS.items() if v == DEFAULT_STEPS),
                                    label="💎 Quality / speed",
                                    info="More steps = sharper detail; time grows linearly")
                seed = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = random)")

    with gr.Accordion("🖼️ Image → Video (start / end frame)", open=False):
        with gr.Row():
            start_img = gr.Image(type="pil", label="Start frame (optional)", height=220)
            end_img = gr.Image(type="pil", label="End frame (optional)", height=220)
        gr.Markdown("*Start only = animate the image. Start + end = first/last-frame interpolation. "
                    "Describe the motion in the prompt; the image fixes the look.*")

    with gr.Accordion("🧑‍🎤 Reference → Video (characters, objects, voice)", open=False, visible=REFERENCES_SUPPORTED):
        with gr.Row():
            ref1 = gr.Image(type="pil", label="Reference 1", height=200)
            ref2 = gr.Image(type="pil", label="Reference 2", height=200)
            ref3 = gr.Image(type="pil", label="Reference 3", height=200)
        with gr.Row():
            ref_names = gr.Textbox(label="Who / what is in each reference (comma separated, optional)",
                                   placeholder="the woman in red, the robot dog, the old castle")
            voice = gr.Audio(type="filepath", label="🎙️ Voice reference (optional, 2-15 s)")
        gr.Markdown("*References keep identity/appearance but place them in a new scene. A voice reference clones "
                    "the voice of Reference 1 for the spoken line.*")

    with gr.Accordion("🎨 LoRAs (style / motion add-ons)", open=False):
        with gr.Row():
            lora1 = gr.Dropdown(LORA_CHOICES, value=NO_LORA, label="LoRA 1")
            s1 = gr.Slider(0.0, 1.5, value=1.0, step=0.05, label="Strength 1")
        lora1_info = gr.Markdown("")
        with gr.Row():
            lora2 = gr.Dropdown(LORA_CHOICES, value=NO_LORA, label="LoRA 2")
            s2 = gr.Slider(0.0, 1.5, value=1.0, step=0.05, label="Strength 2")
        lora2_info = gr.Markdown("")
        with gr.Row():
            custom_url = gr.Textbox(label="Custom LoRA URL (Hugging Face .safetensors link)", scale=3,
                                    placeholder="https://huggingface.co/user/repo/resolve/main/my_h3_lora.safetensors")
            custom_file = gr.File(label="...or upload a .safetensors", file_types=[".safetensors"], scale=2)
            s3 = gr.Slider(0.0, 1.5, value=1.0, step=0.05, label="Custom strength", scale=1)
        auto_trigger = gr.Checkbox(value=True, label="Automatically add each LoRA's trigger words to the prompt")

    with gr.Accordion("⚙️ Advanced", open=False):
        with gr.Row():
            sampler = gr.Dropdown(["euler", "res_multistep"], value=DEFAULT_SAMPLER, label="Sampler",
                                  info="euler: what PDMD / DMAD were run with; res_multistep: crisper audio with the fused turbo")
            shift = gr.Slider(1.0, 16.0, value=12.0, step=0.5, label="Flow shift (video)")
            custom_steps = gr.Slider(0, 12, value=0, step=1, label="Custom steps (0 = use the preset)")

    with gr.Row():
        gen_btn = gr.Button("🎬 Generate Video", variant="primary", size="lg")
        stop_btn = gr.Button("🛑 Stop", size="lg", elem_id="stop-btn")
        clear_btn = gr.Button("🗑️ Clear", size="lg", elem_id="clear-btn")
    video_out = gr.Video(label="🎥 Generated video (with audio)")
    latest_btn = gr.Button("📂 Load latest video (use if the UI lost its connection)")
    status_out = gr.Textbox(label="ℹ️ Status", lines=6, interactive=False)
    seen_job = gr.State(0)
    poll_timer = gr.Timer(2.0)

    gr.Examples(EXAMPLES, inputs=[prompt, dialogue, language, sound, music], label="✨ Example prompts")
    gr.HTML(FOOTER_HTML)

    lora1.change(lora_info, lora1, [s1, lora1_info])
    lora2.change(lora_info, lora2, [s2, lora2_info])
    gen_btn.click(
        start_generation,
        inputs=[prompt, dialogue, language, sound, music, start_img, end_img, ref1, ref2, ref3, ref_names, voice,
                resolution, aspect, duration, steps, sampler, shift, seed, lora1, s1, lora2, s2, custom_url,
                custom_file, s3, auto_trigger, custom_steps],
        outputs=[video_out, status_out], queue=False)
    poll_timer.tick(poll_job, inputs=seen_job, outputs=[video_out, status_out, seen_job], queue=False,
                    show_progress="hidden")
    stop_btn.click(stop_generation, outputs=status_out, queue=False)
    latest_btn.click(load_latest_video, outputs=[video_out, status_out], queue=False)
    clear_btn.click(lambda: (None, "Ready.", "", "", "", "", None, None, None, None, None, "", None, -1),
                    outputs=[video_out, status_out, prompt, dialogue, sound, music, start_img, end_img, ref1, ref2, ref3,
                             ref_names, voice, seed], queue=False)


# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
# Started without preexec_fn: that would force a real fork() of this process (Python otherwise uses vfork), and a
# fork has to copy every pinned page at once (see _dont_fork). Step 2 stops leftover tunnels before each start.
# The link only works once (a) cloudflared has registered a connection with Cloudflare's edge and (b) the random
# *.trycloudflare.com name is published in DNS. cloudflared tries QUIC (UDP) first, which Kaggle's network often blocks,
# and only falls back to HTTP/2 after a while: hence --protocol http2. And a browser that asks for the name before it
# exists caches the "not found" answer for minutes, so the link is shown as ready only after both checks pass.
def _dns_published(host):
    """Ask Cloudflare's DNS-over-HTTPS directly: no local / Kaggle resolver caches a negative answer.
    True / False, or None when the lookup itself is unavailable."""
    try:
        req = urllib.request.Request(f"https://cloudflare-dns.com/dns-query?name={host}&type=A",
                                     headers={"accept": "application/dns-json"})
        import json
        return bool(json.loads(urllib.request.urlopen(req, timeout=5).read()).get("Answer"))
    except Exception:
        return None


def start_cloudflare_tunnel(port):
    import stat
    found = {"url": None, "registered": False, "ready": False}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                                   "--protocol", "http2"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        import atexit
        atexit.register(tunnel.kill)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)  # skip cloudflared's API host
            if match and not found["url"]:
                found["url"] = match.group(0)
            if "Registered tunnel connection" in line:
                found["registered"] = True

    def _wait_ready():
        deadline, registered_at = time.time() + 300, None
        while time.time() < deadline:
            if found["registered"] and registered_at is None:
                registered_at = time.time()
            published = _dns_published(found["url"].split("//", 1)[1]) if found["url"] and registered_at else False
            if published or (published is None and time.time() - registered_at > 15):
                found["ready"] = True
                print(f"\n✅ Cloudflare link is ready: {found['url']}\n")
                sys.stdout.flush()
                return
            time.sleep(3)
        print("⚠️ The Cloudflare link did not come up within 5 minutes; use the Gradio share link.")

    threading.Thread(target=_watch, daemon=True).start()
    threading.Thread(target=_wait_ready, daemon=True).start()
    return found



print("🌐 Starting the public links (Cloudflare + Gradio share)...")
sys.stdout.flush()
cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20)
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
            prevent_thread_lock=True, quiet=True, ssr_mode=False, max_threads=8, allowed_paths=[OUTPUT_DIR])
from fastapi.responses import FileResponse, JSONResponse
from fastapi.routing import APIRoute


def _progress_route():
    end = PROGRESS["finished"] or time.time()
    elapsed = end - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})


def _video_route():
    """The newest video as a plain file (plays in the browser, with a download option): no Gradio involved."""
    videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.mp4")), key=os.path.getmtime)
    if not videos:
        return JSONResponse({"error": "no video yet"}, status_code=404)
    name = os.path.basename(videos[-1])
    return FileResponse(videos[-1], media_type="video/mp4", headers={"Content-Disposition": f'inline; filename="{name}"',
                                                                      "Cache-Control": "no-store"})


for _path, _route in (("/aiq_progress", _progress_route), ("/aiq_video", _video_route)):
    demo.app.router.routes.insert(0, APIRoute(_path, _route, methods=["GET"]))  # ahead of Gradio's routes
for _ in range(60):  # give the tunnel up to ~30 s to come up before printing the links
    if cf["ready"]:
        break
    time.sleep(0.5)

print("\n" + "=" * 70)
print("🎬 MiniMax H3 Turbo is LIVE!")
print("=" * 70)
if cf["ready"]:
    print(f"🌐 Cloudflare tunnel: {cf['url']}")
else:
    print(f"🌐 Cloudflare tunnel: ⏳ warming up{' (' + cf['url'] + ')' if cf['url'] else ''} - wait for the "
          f"'✅ Cloudflare link is ready' line before opening it; opening it too early can keep it unreachable for minutes")
print(f"🔗 Gradio share:      {getattr(demo, 'share_url', None) or '(unavailable, use the Cloudflare link)'}")
print(f"💾 Videos are saved to {OUTPUT_DIR} (Kaggle Output panel)")
print(f"🎞️ Latest video, without the UI: <link>/aiq_video (e.g. {cf['url'] or 'https://...trycloudflare.com'}/aiq_video)")
print("=" * 70 + "\n")
sys.stdout.flush()
while True:  # keep the server process alive; generation logs stream to the notebook cell
    time.sleep(3600)
'''

with open("run_h3.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)
print("✅ run_h3.py written.")

import json
if os.path.exists("h3_settings.json"):  # Step 1's choices, for a Step 2 re-run after a kernel restart
    with open("h3_settings.json") as f:
        for key, value in json.load(f).items():
            os.environ.setdefault(key, value)

IGNORE = ("absl::InitializeLog", "Unable to register cu", "computation placer already registered", "MessageFactory",
          "GetPrototype", "Attempting to register factory", "UserWarning", "warnings.warn", "FutureWarning")
env = dict(os.environ, PYTHONWARNINGS="ignore", TQDM_DISABLE="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

print("🚀 Starting the MiniMax H3 engine...\n")
import psutil

def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~23 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_h3.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()

# Step 1's downloads leave freed buffers in this kernel process: hand them back to the OS before the engine starts.
import ctypes
import gc
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass
_kernel_gb = psutil.Process().memory_info().rss / 1024**3
_ram = psutil.virtual_memory()
print(f"🧠 RAM before the engine starts: {(_ram.total - _ram.available) / 1024**3:.1f} GB in use "
      f"(this notebook kernel: {_kernel_gb:.1f} GB)")
if _kernel_gb > 2.0:
    print("   ⚠️ The kernel itself holds a lot of RAM. If the engine runs out of memory: Run → Restart, then run Step 2 only.")


def _die_with_kernel():
    # Linux: the engine gets SIGKILL when this notebook kernel dies or restarts (no orphan holding the GPUs).
    try:
        import ctypes
        import signal
        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGKILL)  # PR_SET_PDEATHSIG
    except Exception:
        pass


proc = subprocess.Popen([sys.executable, "-u", "run_h3.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                        preexec_fn=_die_with_kernel)
import threading
_peak = {"ram": 0.0, "engine": 0.0}


def _watch_ram():  # peak RAM while the engine runs, reported if it dies
    engine = psutil.Process(proc.pid)
    while proc.poll() is None:
        try:
            vm = psutil.virtual_memory()
            _peak["ram"] = max(_peak["ram"], (vm.total - vm.available) / 1024**3)
            _peak["engine"] = max(_peak["engine"], engine.memory_info().rss / 1024**3)
        except Exception:
            pass
        time.sleep(0.5)


import time
threading.Thread(target=_watch_ram, daemon=True).start()
# Read raw bytes so carriage returns survive: the engine redraws its live progress line in place, and a text-mode
# pipe would turn every redraw into a new line.
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()  # text after the last line break / carriage return
        out = "".join(part for part in parts if not any(pattern in part for pattern in IGNORE))
        if pending and not any(pattern in pending for pattern in IGNORE):
            out, pending = out + pending, ""  # show the live progress line right away
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        import signal
        how = f"killed by {signal.Signals(-rc).name}" if rc < 0 else f"exit code {rc}"
        hint = (" - the system killed it, which almost always means it ran out of RAM" if rc == -9 else
                " - a crash inside a native library (see the traceback above)" if rc in (-11, -6) else "")
        print(f"\n❌ The engine stopped ({how}){hint}.\n   Peak RAM while it ran: {_peak['ram']:.1f} GB in use of "
              f"{psutil.virtual_memory().total / 1024**3:.1f} GB (engine process {_peak['engine']:.1f} GB).")
except KeyboardInterrupt:
    proc.kill()  # frees the GPUs and RAM right away
    proc.wait()
    print("🛑 Engine stopped. Re-run this cell to start it again.")

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---